In [1]:
# ── Cell 0: Install Dependencies ──────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "torch", "torchaudio",
    "numpy", "pandas", "scikit-learn",
    "librosa", "soundfile",
    "tqdm", "matplotlib", "seaborn",
)
## Imports and Global Utilities
import os, sys, random, json, time, warnings, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import soundfile as sf
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    balanced_accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc as sk_auc
)
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ── General utilities ─────────────────────────────────────────────────────────
def to_numpy(t):
    return t.detach().cpu().numpy() if isinstance(t, torch.Tensor) else np.array(t)

def flatten_dict(d, parent_key="", sep="/"):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

In [2]:
## Logging
import logging
from datetime import datetime

LOG_DIR = Path("/kaggle/working/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = LOG_DIR / f"run_{run_id}.logger.info"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("mega_ia")

# ── Experiment result accumulator ─────────────────────────────────────────────
experiment_log = []

def log_result(exp_name: str, metrics: dict):
    entry = {"experiment": exp_name, "timestamp": run_id, **flatten_dict(metrics)}
    experiment_log.append(entry)
    logger.info(f"[{exp_name}] {metrics}")

def save_experiment_log(path: str = None):
    out = Path(path) if path else LOG_DIR / f"results_{run_id}.json"
    with open(out, "w") as f:
        json.dump(experiment_log, f, indent=2, default=str)
    logger.info(f"Experiment logger.info saved → {out}")

logger.info(f"Logging initialised | run_id={run_id} | logger.info={log_file}")

2026-05-09 05:26:12,554 | INFO | Logging initialised | run_id=20260509_052612 | logger.info=logs/run_20260509_052612.logger.info


In [3]:
## Paths and Configurattions
from pathlib import Path


VALPATH          = Path("/kaggle/input/datasets/aww4bahmad/dataseet/datasets/validation_dataset")
VAL_REAL_DIR     = VALPATH / "real"
VAL_FAKE_DIR     = VALPATH / "fake"
VAL_META_CSV     = Path("/kaggle/input/datasets/aww4bahmad/dataseet/datasets/validation_dataset/meta.csv")

LA_TEST_META        = Path("/kaggle/input/datasets/aww4bahmad/dataseet/datasets/la_test/meta.csv")
IN_THE_WILD_META    = Path("/kaggle/input/datasets/aww4bahmad/dataseet/datasets/in_the_wild_test/meta.csv")
DEEPFAKE_EVALS_META = Path("/kaggle/input/datasets/aww4bahmad/dataseet/datasets/Deepfake-Eval-2024/audio-metadata-publish.csv")

CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "baseline":   CKPT_DIR / "rawnet2_baseline.pth",
    "mega_ia_v1": CKPT_DIR / "rawnet2_mega_ia_v1.pth",
    "mega_ia_v2": CKPT_DIR / "rawnet2_mega_ia_v2.pth",
    "la_parent":  Path("/kaggle/input/models/aww4bahmad/rawnet/pytorch/default/1/best_model.pth"),
    "itw_parent": Path("/kaggle/input/models/aww4bahmad/rawnet/pytorch/default/1/best_auc.pth"),
}

SAMPLE_RATE  = 16000
MAX_DURATION = 4
MAX_SAMPLES  = SAMPLE_RATE * MAX_DURATION
BATCH_SIZE   = 32
NUM_WORKERS  = 4
NUM_EPOCHS   = 50
LR           = 1e-4
SEED         = 42

for p in [VALPATH, VAL_REAL_DIR, VAL_FAKE_DIR]:
    if not p.exists(): logger.warning(f'Path not found: {p}')
    else: logger.info(f'OK: {p}')

2026-05-09 05:26:12,570 | INFO | OK: /kaggle/input/datasets/aww4bahmad/dataseet/datasets/validation_dataset
2026-05-09 05:26:12,572 | INFO | OK: /kaggle/input/datasets/aww4bahmad/dataseet/datasets/validation_dataset/real
2026-05-09 05:26:12,574 | INFO | OK: /kaggle/input/datasets/aww4bahmad/dataseet/datasets/validation_dataset/fake


In [4]:
## Device Setup (Multi-GPU)
import torch

N_GPUS = torch.cuda.device_count()
if N_GPUS >= 2:
    DEVICE = torch.device("cuda:0")
    DEVICE2 = torch.device("cuda:1")
    torch.backends.cudnn.benchmark = True
    for i in range(N_GPUS):
        props = torch.cuda.get_device_properties(i)
        logger.info(f"CUDA:{i} — {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")
    logger.info(f"Multi-GPU mode: {N_GPUS} GPUs detected. Primary=cuda:0, Secondary=cuda:1")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE2 = DEVICE
    torch.backends.cudnn.benchmark = True
    logger.info(f"CUDA device: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    logger.warning("Only 1 GPU detected — running single-GPU mode.")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DEVICE2 = DEVICE
    logger.info("Device: Apple MPS")
else:
    DEVICE = torch.device("cpu")
    DEVICE2 = DEVICE
    logger.info("Device: CPU")

logger.info(f"Active devices → primary={DEVICE}, secondary={DEVICE2}")

2026-05-09 05:26:12,883 | INFO | CUDA:0 — Tesla T4 | VRAM: 15.6 GB
2026-05-09 05:26:12,883 | INFO | CUDA:1 — Tesla T4 | VRAM: 15.6 GB
2026-05-09 05:26:12,884 | INFO | Multi-GPU mode: 2 GPUs detected. Primary=cuda:0, Secondary=cuda:1
2026-05-09 05:26:12,885 | INFO | Active devices → primary=cuda:0, secondary=cuda:1


In [5]:
## RawNet2 Architecture
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import numpy as np
import math
from torch.utils import data
from collections import OrderedDict
from torch.nn.parameter import Parameter
from torch.autograd import Variable
import pickle
import random


class SincConv(nn.Module):
    @staticmethod
    def to_mel(hz):
        return 2595 * np.log10(1 + hz / 700)

    @staticmethod
    def to_hz(mel):
        return 700 * (10 ** (mel / 2595) - 1)


    def __init__(self, device,out_channels, kernel_size,in_channels=1,sample_rate=16000,
                 stride=1, padding=0, dilation=1, bias=False, groups=1,freq_scale='Mel'):

        super(SincConv,self).__init__()


        if in_channels != 1:
            
            msg = "SincConv only support one input channel (here, in_channels = {%i})" % (in_channels)
            raise ValueError(msg)
        
        self.out_channels = out_channels+1
        self.kernel_size = kernel_size
        self.sample_rate=sample_rate

        # Forcing the filters to be odd (i.e, perfectly symmetrics)
        if kernel_size%2==0:
            self.kernel_size=self.kernel_size+1

        self.device=device   
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        
        if bias:
            raise ValueError('SincConv does not support bias.')
        if groups > 1:
            raise ValueError('SincConv does not support groups.')
        
        
        # initialize filterbanks using Mel scale
        NFFT = 512
        f=int(self.sample_rate/2)*np.linspace(0,1,int(NFFT/2)+1)


        if freq_scale == 'Mel':
            fmel=self.to_mel(f) # Hz to mel conversion
            fmelmax=np.max(fmel)
            fmelmin=np.min(fmel)
            filbandwidthsmel=np.linspace(fmelmin,fmelmax,self.out_channels+2)
            filbandwidthsf=self.to_hz(filbandwidthsmel) # Mel to Hz conversion
            self.freq=filbandwidthsf[:self.out_channels]

        elif freq_scale == 'Inverse-mel':
            fmel=self.to_mel(f) # Hz to mel conversion
            fmelmax=np.max(fmel)
            fmelmin=np.min(fmel)
            filbandwidthsmel=np.linspace(fmelmin,fmelmax,self.out_channels+2)
            filbandwidthsf=self.to_hz(filbandwidthsmel) # Mel to Hz conversion
            self.mel=filbandwidthsf[:self.out_channels]
            self.freq=np.abs(np.flip(self.mel)-1) ## invert mel scale

        
        else:
            fmelmax=np.max(f)
            fmelmin=np.min(f)
            filbandwidthsmel=np.linspace(fmelmin,fmelmax,self.out_channels+2)
            self.freq=filbandwidthsmel[:self.out_channels]
        
        self.hsupp=torch.arange(-(self.kernel_size-1)/2, (self.kernel_size-1)/2+1)
        self.band_pass=torch.zeros(self.out_channels-1,self.kernel_size)
    
       
        
    def forward(self,x):
        for i in range(len(self.freq)-1):
            fmin=self.freq[i]
            fmax=self.freq[i+1]
            hHigh=(2*fmax/self.sample_rate)*np.sinc(2*fmax*self.hsupp/self.sample_rate)
            hLow=(2*fmin/self.sample_rate)*np.sinc(2*fmin*self.hsupp/self.sample_rate)
            hideal=hHigh-hLow
            
            self.band_pass[i,:]=Tensor(np.hamming(self.kernel_size))*Tensor(hideal)
        
        band_pass_filter=self.band_pass.to(self.device)

        self.filters = (band_pass_filter).view(self.out_channels-1, 1, self.kernel_size)
        
        return F.conv1d(x, self.filters, stride=self.stride,
                        padding=self.padding, dilation=self.dilation,
                         bias=None, groups=1)


        
class Residual_block(nn.Module):
    def __init__(self, nb_filts, first = False):
        super(Residual_block, self).__init__()
        self.first = first
        
        if not self.first:
            self.bn1 = nn.BatchNorm1d(num_features = nb_filts[0])
        
        self.lrelu = nn.LeakyReLU(negative_slope=0.3)
        
        self.conv1 = nn.Conv1d(in_channels = nb_filts[0],
			out_channels = nb_filts[1],
			kernel_size = 3,
			padding = 1,
			stride = 1)
        
        self.bn2 = nn.BatchNorm1d(num_features = nb_filts[1])
        self.conv2 = nn.Conv1d(in_channels = nb_filts[1],
			out_channels = nb_filts[1],
			padding = 1,
			kernel_size = 3,
			stride = 1)
        
        if nb_filts[0] != nb_filts[1]:
            self.downsample = True
            self.conv_downsample = nn.Conv1d(in_channels = nb_filts[0],
				out_channels = nb_filts[1],
				padding = 0,
				kernel_size = 1,
				stride = 1)
            
        else:
            self.downsample = False
        self.mp = nn.MaxPool1d(3)
        
    def forward(self, x):
        identity = x
        if not self.first:
            out = self.bn1(x)
            out = self.lrelu(out)
        else:
            out = x
            
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.lrelu(out)
        out = self.conv2(out)
        
        if self.downsample:
            identity = self.conv_downsample(identity)
            
        out += identity
        out = self.mp(out)
        return out





class RawNet(nn.Module):
    def __init__(self, d_args, device):
        super(RawNet, self).__init__()

        
        self.device=device

        self.Sinc_conv=SincConv(device=self.device,
			out_channels = d_args['filts'][0],
			kernel_size = d_args['first_conv'],
                        in_channels = d_args['in_channels'],freq_scale='Mel'
        )
        
        self.first_bn = nn.BatchNorm1d(num_features = d_args['filts'][0])
        self.selu = nn.SELU(inplace=True)
        self.block0 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][1], first = True))
        self.block1 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][1]))
        self.block2 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][2]))
        # Compute [out, out] config for blocks 3-5 WITHOUT mutating d_args in-place.
        # The original code did `d_args['filts'][2][0] = d_args['filts'][2][1]` which
        # permanently corrupted d_args for every subsequent RawNet() call in the session.
        filts2_after = [d_args['filts'][2][1], d_args['filts'][2][1]]
        self.block3 = nn.Sequential(Residual_block(nb_filts = filts2_after))
        self.block4 = nn.Sequential(Residual_block(nb_filts = filts2_after))
        self.block5 = nn.Sequential(Residual_block(nb_filts = filts2_after))
        self.avgpool = nn.AdaptiveAvgPool1d(1)

        self.fc_attention0 = self._make_attention_fc(in_features = d_args['filts'][1][-1],
            l_out_features = d_args['filts'][1][-1])
        self.fc_attention1 = self._make_attention_fc(in_features = d_args['filts'][1][-1],
            l_out_features = d_args['filts'][1][-1])
        self.fc_attention2 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])
        self.fc_attention3 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])
        self.fc_attention4 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])
        self.fc_attention5 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])

        self.bn_before_gru = nn.BatchNorm1d(num_features = d_args['filts'][2][-1])
        self.gru = nn.GRU(input_size = d_args['filts'][2][-1],
			hidden_size = d_args['gru_node'],
			num_layers = d_args['nb_gru_layer'],
			batch_first = True)

        
        self.fc1_gru = nn.Linear(in_features = d_args['gru_node'],
			out_features = d_args['nb_fc_node'])
       
        self.fc2_gru = nn.Linear(in_features = d_args['nb_fc_node'],
			out_features = d_args['nb_classes'],bias=True)
			
       
        self.sig = nn.Sigmoid()
        
        
    def forward(self, x, y = None,is_test=False):
        
        
        nb_samp = x.shape[0]
        len_seq = x.shape[1]
        x=x.view(nb_samp,1,len_seq)
        
        x = self.Sinc_conv(x)    # Fixed sinc filters convolution
        x = F.max_pool1d(torch.abs(x), 3)
        x = self.first_bn(x)
        x = self.selu(x)
        
        x0 = self.block0(x)
        y0 = self.avgpool(x0).view(x0.size(0), -1) # torch.Size([batch, filter])
        y0 = self.fc_attention0(y0)
        y0 = self.sig(y0).view(y0.size(0), y0.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x0 * y0 + y0  # (batch, filter, time) x (batch, filter, 1)
        

        x1 = self.block1(x)
        y1 = self.avgpool(x1).view(x1.size(0), -1) # torch.Size([batch, filter])
        y1 = self.fc_attention1(y1)
        y1 = self.sig(y1).view(y1.size(0), y1.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x1 * y1 + y1 # (batch, filter, time) x (batch, filter, 1)

        x2 = self.block2(x)
        y2 = self.avgpool(x2).view(x2.size(0), -1) # torch.Size([batch, filter])
        y2 = self.fc_attention2(y2)
        y2 = self.sig(y2).view(y2.size(0), y2.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x2 * y2 + y2 # (batch, filter, time) x (batch, filter, 1)

        x3 = self.block3(x)
        y3 = self.avgpool(x3).view(x3.size(0), -1) # torch.Size([batch, filter])
        y3 = self.fc_attention3(y3)
        y3 = self.sig(y3).view(y3.size(0), y3.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x3 * y3 + y3 # (batch, filter, time) x (batch, filter, 1)

        x4 = self.block4(x)
        y4 = self.avgpool(x4).view(x4.size(0), -1) # torch.Size([batch, filter])
        y4 = self.fc_attention4(y4)
        y4 = self.sig(y4).view(y4.size(0), y4.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x4 * y4 + y4 # (batch, filter, time) x (batch, filter, 1)

        x5 = self.block5(x)
        y5 = self.avgpool(x5).view(x5.size(0), -1) # torch.Size([batch, filter])
        y5 = self.fc_attention5(y5)
        y5 = self.sig(y5).view(y5.size(0), y5.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x5 * y5 + y5 # (batch, filter, time) x (batch, filter, 1)

        x = self.bn_before_gru(x)
        x = self.selu(x)
        x = x.permute(0, 2, 1)     #(batch, filt, time) >> (batch, time, filt)
        self.gru.flatten_parameters()
        x, _ = self.gru(x)
        x = x[:,-1,:]
        x = self.fc1_gru(x)
        x = self.fc2_gru(x)

        if not is_test:
            output = x
            return output

        else:
            output=F.softmax(x,dim=1)
            return output
      
       
        

    def _make_attention_fc(self, in_features, l_out_features):

        l_fc = []
        
        l_fc.append(nn.Linear(in_features = in_features,
			        out_features = l_out_features))

        

        return nn.Sequential(*l_fc)


    def _make_layer(self, nb_blocks, nb_filts, first = False):
        layers = []
        #def __init__(self, nb_filts, first = False):
        for i in range(nb_blocks):
            first = first if i == 0 else False
            layers.append(Residual_block(nb_filts = nb_filts,
				first = first))
            if i == 0: nb_filts[0] = nb_filts[1]
            
        return nn.Sequential(*layers)

    def summary(self, input_size, batch_size=-1, device="cuda", print_fn = None):
        if print_fn == None: printfn = print
        model = self
        
        def register_hook(module):
            def hook(module, input, output):
                class_name = str(module.__class__).split(".")[-1].split("'")[0]
                module_idx = len(summary)
                
                m_key = "%s-%i" % (class_name, module_idx + 1)
                summary[m_key] = OrderedDict()
                summary[m_key]["input_shape"] = list(input[0].size())
                summary[m_key]["input_shape"][0] = batch_size
                if isinstance(output, (list, tuple)):
                    summary[m_key]["output_shape"] = [
						[-1] + list(o.size())[1:] for o in output
					]
                else:
                    summary[m_key]["output_shape"] = list(output.size())
                    if len(summary[m_key]["output_shape"]) != 0:
                        summary[m_key]["output_shape"][0] = batch_size
                        
                params = 0
                if hasattr(module, "weight") and hasattr(module.weight, "size"):
                    params += torch.prod(torch.LongTensor(list(module.weight.size())))
                    summary[m_key]["trainable"] = module.weight.requires_grad
                if hasattr(module, "bias") and hasattr(module.bias, "size"):
                    params += torch.prod(torch.LongTensor(list(module.bias.size())))
                summary[m_key]["nb_params"] = params
                
            if (
				not isinstance(module, nn.Sequential)
				and not isinstance(module, nn.ModuleList)
				and not (module == model)
			):
                hooks.append(module.register_forward_hook(hook))
                
        device = device.lower()
        assert device in [
			"cuda",
			"cpu",
		], "Input device is not valid, please specify 'cuda' or 'cpu'"
        
        if device == "cuda" and torch.cuda.is_available():
            dtype = torch.cuda.FloatTensor
        else:
            dtype = torch.FloatTensor
        if isinstance(input_size, tuple):
            input_size = [input_size]
        x = [torch.rand(2, *in_size).type(dtype) for in_size in input_size]
        summary = OrderedDict()
        hooks = []
        model.apply(register_hook)
        model(*x)
        for h in hooks:
            h.remove()

In [6]:
## Dataset Classes
import warnings
from torch.utils.data import Dataset

def pad_random(x: np.ndarray, max_len: int = 64600) -> np.ndarray:
    x_len = x.shape[0]
    if x_len > max_len:
        stt = np.random.randint(x_len - max_len)
        return x[stt:stt + max_len]
    return np.tile(x, (int(max_len / x_len) + 1))[:max_len]

class ValidationDataset(Dataset):
    """
    Local validation dataset.
    CSV columns: filename, label ('real' | 'fake')
    Files are in root_dir/real/ and root_dir/fake/ subdirs.
    """
    TARGET_SR = 16000

    def __init__(self, root_dir: str):
        self.root_dir = root_dir
        df = pd.read_csv(os.path.join(root_dir, 'meta.csv'))
        self.df = df.reset_index(drop=True)
        self.n_fake = int((self.df['label'] == 'fake').sum())
        self.n_real = int((self.df['label'] == 'real').sum())

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        subdir = row['label']   # 'real' or 'fake' — matches the subdir name exactly
        path  = os.path.join(self.root_dir, subdir, row['filename'])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            x, _ = librosa.load(path, sr=self.TARGET_SR, mono=True)
        label = 1 if row['label'] == 'real' else 0
        return torch.FloatTensor(pad_random(x)), label


logger.info('ValidationDataset patched (real/fake subdirs).')

class LATestDataset(Dataset):
    """
    ASVspoof LA test set.
    CSV columns: Filename, Ground Truth
    Ground Truth: 'Real' | 'Fake'
    """
    TARGET_SR = 16000

    def __init__(self, meta_csv: str):
        self.root_dir = os.path.dirname(meta_csv)
        df = pd.read_csv(meta_csv)
        self.df = df.reset_index(drop=True)
        self.n_real = int((self.df['Ground Truth'] == 'Real').sum())
        self.n_fake = int((self.df['Ground Truth'] == 'Fake').sum())

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.root_dir, str(row['Filename']))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            x, _ = librosa.load(path, sr=self.TARGET_SR, mono=True)
        label = 1 if row['Ground Truth'] == 'Real' else 0
        return torch.FloatTensor(pad_random(x)), label


class InTheWildDataset(Dataset):
    """
    In-the-Wild test set.
    CSV columns: file, speaker, label
    label: 'bona-fide' | 'spoof'
    """
    TARGET_SR = 16000

    def __init__(self, meta_csv: str):
        self.root_dir = os.path.dirname(meta_csv)
        df = pd.read_csv(meta_csv)
        self.df = df.reset_index(drop=True)
        self.n_real = int((self.df['label'] == 'bona-fide').sum())
        self.n_fake = int((self.df['label'] == 'spoof').sum())

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.root_dir, str(row['file']))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            x, _ = librosa.load(path, sr=self.TARGET_SR, mono=True)
        label = 1 if row['label'] == 'bona-fide' else 0
        return torch.FloatTensor(pad_random(x)), label


class DeepfakeEvalsDataset(Dataset):
    """
    Deepfake-Evals-2024.
    CSV columns: Filename, Date, Ground Truth, Public Comments, Finetuning Set
    Ground Truth: 'Real' | 'Fake'
    Audio files are under root_dir/audio-data/<Filename>
    """
    TARGET_SR = 16000
    AUDIO_SUBDIR = "audio-data"

    def __init__(self, root_dir: str, cache_dir: str = None, indices=None):
        self.root_dir  = root_dir
        self.audio_dir = os.path.join(root_dir, self.AUDIO_SUBDIR)
        df = pd.read_csv(os.path.join(root_dir, 'audio-metadata-publish.csv'))
        df = df[df['Filename'].notna()].copy()
        df['Filename'] = df['Filename'].astype(str).str.strip()

        cache_path = os.path.join(cache_dir, 'deepfake_valid_indices.json') if cache_dir else None

        if cache_path and os.path.exists(cache_path):
            with open(cache_path) as f:
                cached = json.load(f)
            good = cached['good_indices']
            logger.info(f'Loaded cached file list ({len(good)} valid, {cached.get("skip_count",0)} skipped).')
        else:
            logger.info('Pre-flighting Deepfake-Evals files (runs once, then cached)...')
            good, skip = [], []
            total = len(df)
            for pos, (i, row) in enumerate(df.iterrows()):
                if pos % 100 == 0:
                    logger.info(f'  Pre-flight: {pos}/{total}...')
                path = os.path.join(self.audio_dir, row['Filename'])
                if not os.path.isfile(path):
                    skip.append(row['Filename']); continue
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        librosa.load(path, sr=None, mono=True, duration=0.1)
                    good.append(i)
                except Exception:
                    skip.append(row['Filename'])
            if cache_path:
                with open(cache_path, 'w') as f:
                    json.dump({'good_indices': good, 'skip_count': len(skip)}, f)
                logger.info(f'Valid file list cached → {cache_path}')

        self.df = df.loc[good].reset_index(drop=True)
        if indices is not None:
            self.df = self.df.loc[indices].reset_index(drop=True)

        self.n_real = int((self.df['Ground Truth'] == 'Real').sum())
        self.n_fake = int((self.df['Ground Truth'] == 'Fake').sum())

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, str(row['Filename']))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            x, _ = librosa.load(path, sr=self.TARGET_SR, mono=True)
        label = 1 if row['Ground Truth'] == 'Real' else 0
        return torch.FloatTensor(pad_random(x)), label


logger.info('Dataset classes defined (all using librosa).')


def split_deepfake_evals(root_dir: str, cache_dir: str,
                         val_frac: float = 0.15,
                         seed: int = SEED):
    """
    Return three disjoint DeepfakeEvalsDataset instances:
        full_ds   — 100% (all valid files)
        val_ds    — 15%  balanced: equal Real/Fake counts  (7.5% each)
        test_ds   — 85%  remainder (all files not in val)

    Constraint: val set is balanced — Real count == Fake count.
    The smaller of (target_real, target_fake) determines val size per class.
    """
    rng = np.random.default_rng(seed)

    # Build full dataset first to get the validated df
    full_ds = DeepfakeEvalsDataset(root_dir, cache_dir=cache_dir)
    df = full_ds.df.copy()

    real_idx = df.index[df['Ground Truth'] == 'Real'].tolist()
    fake_idx = df.index[df['Ground Truth'] == 'Fake'].tolist()

    n_total      = len(df)
    n_val_target = int(n_total * val_frac)          # 15% of total
    n_per_class  = n_val_target // 2                # equal split
    # Clamp to available counts
    n_per_class  = min(n_per_class, len(real_idx), len(fake_idx))

    rng.shuffle(real_idx := np.array(real_idx))
    rng.shuffle(fake_idx := np.array(fake_idx))

    val_real = real_idx[:n_per_class].tolist()
    val_fake = fake_idx[:n_per_class].tolist()
    val_idx  = val_real + val_fake

    val_idx_set  = set(val_idx)
    test_idx     = [i for i in df.index.tolist() if i not in val_idx_set]

    val_ds  = DeepfakeEvalsDataset(root_dir, cache_dir=cache_dir, indices=val_idx)
    test_ds = DeepfakeEvalsDataset(root_dir, cache_dir=cache_dir, indices=test_idx)

    logger.info(f'DFE full : {len(full_ds):,}  ({full_ds.n_real:,} real / {full_ds.n_fake:,} fake)')
    logger.info(f'DFE val  : {len(val_ds):,}  ({val_ds.n_real:,} real / {val_ds.n_fake:,} fake)  [{len(val_ds)/len(full_ds)*100:.1f}%]')
    logger.info(f'DFE test : {len(test_ds):,}  ({test_ds.n_real:,} real / {test_ds.n_fake:,} fake)  [{len(test_ds)/len(full_ds)*100:.1f}%]')
    assert len(val_idx_set & set(test_idx)) == 0, 'Disjoint constraint violated!'
    assert val_ds.n_real == val_ds.n_fake,         'Val balance constraint violated!'

    return full_ds, val_ds, test_ds


logger.info('Dataset classes defined.')

2026-05-09 05:26:12,954 | INFO | ValidationDataset patched (real/fake subdirs).
2026-05-09 05:26:12,956 | INFO | Dataset classes defined (all using librosa).
2026-05-09 05:26:12,958 | INFO | Dataset classes defined.


In [7]:
## Load Datasets (Full — No Subsampling)
# ── Load Datasets ──────────────────────────────────────────────────────────────
from torch.utils.data import DataLoader

logger.info('Loading validation dataset...')
val_ds = ValidationDataset(str(VALPATH))
logger.info(f'Val:           {len(val_ds):,} samples  ({val_ds.n_real:,} real / {val_ds.n_fake:,} fake)')

logger.info('Loading LA test set...')
la_ds = LATestDataset(str(LA_TEST_META))
logger.info(f'LA test:       {len(la_ds):,} samples  ({la_ds.n_real:,} real / {la_ds.n_fake:,} fake)')

logger.info('Loading In-the-Wild test set...')
itw_ds = InTheWildDataset(str(IN_THE_WILD_META))
logger.info(f'In-the-Wild:   {len(itw_ds):,} samples  ({itw_ds.n_real:,} real / {itw_ds.n_fake:,} fake)')

logger.info('Loading Deepfake-Evals-2024 (full + splits)...')
dfe_root     = str(DEEPFAKE_EVALS_META.parent)
dfe_full_ds, dfe_val_ds, dfe_test_ds = split_deepfake_evals(
    root_dir=dfe_root, cache_dir=str(CKPT_DIR), val_frac=0.15, seed=SEED
)

_pin = (str(DEVICE) == 'cuda')
_nw  = NUM_WORKERS

val_loader      = DataLoader(val_ds,      batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
la_loader       = DataLoader(la_ds,       batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
itw_loader      = DataLoader(itw_ds,      batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
dfe_full_loader = DataLoader(dfe_full_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
dfe_val_loader  = DataLoader(dfe_val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
dfe_test_loader = DataLoader(dfe_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)

logger.info(f'val_loader      : {len(val_loader):,} batches')
logger.info(f'la_loader       : {len(la_loader):,} batches')
logger.info(f'itw_loader      : {len(itw_loader):,} batches')
logger.info(f'dfe_full_loader : {len(dfe_full_loader):,} batches')
logger.info(f'dfe_val_loader  : {len(dfe_val_loader):,} batches')
logger.info(f'dfe_test_loader : {len(dfe_test_loader):,} batches')

# Smoke-test
logger.info('Smoke-testing loaders...')
for name, ldr in [('val', val_loader), ('la', la_loader), ('itw', itw_loader),
                  ('dfe_full', dfe_full_loader), ('dfe_val', dfe_val_loader), ('dfe_test', dfe_test_loader)]:
    xb, yb = next(iter(ldr))
    logger.info(f'{name:>10} batch: {tuple(xb.shape)}  labels={yb[:4].tolist()}')
    del xb, yb
logger.info('All loaders confirmed.')

2026-05-09 05:26:12,972 | INFO | Loading validation dataset...
2026-05-09 05:26:12,988 | INFO | Val:           102 samples  (51 real / 51 fake)
2026-05-09 05:26:12,989 | INFO | Loading LA test set...
2026-05-09 05:26:13,006 | INFO | LA test:       1,000 samples  (500 real / 500 fake)
2026-05-09 05:26:13,006 | INFO | Loading In-the-Wild test set...
2026-05-09 05:26:13,018 | INFO | In-the-Wild:   3,176 samples  (1,588 real / 1,588 fake)
2026-05-09 05:26:13,019 | INFO | Loading Deepfake-Evals-2024 (full + splits)...
2026-05-09 05:26:13,032 | INFO | Pre-flighting Deepfake-Evals files (runs once, then cached)...
2026-05-09 05:26:13,033 | INFO |   Pre-flight: 0/1980...


[src/libmpg123/parse.c:skip_junk():1260] error: Giving up searching valid MPEG header after 65536 bytes of junk.


2026-05-09 05:26:27,290 | INFO |   Pre-flight: 100/1980...
2026-05-09 05:26:28,819 | INFO |   Pre-flight: 200/1980...
2026-05-09 05:26:29,920 | INFO |   Pre-flight: 300/1980...
2026-05-09 05:26:31,409 | INFO |   Pre-flight: 400/1980...
2026-05-09 05:26:32,630 | INFO |   Pre-flight: 500/1980...
2026-05-09 05:26:34,209 | INFO |   Pre-flight: 600/1980...
2026-05-09 05:26:35,310 | INFO |   Pre-flight: 700/1980...
2026-05-09 05:26:37,526 | INFO |   Pre-flight: 800/1980...
2026-05-09 05:26:39,131 | INFO |   Pre-flight: 900/1980...
2026-05-09 05:26:40,600 | INFO |   Pre-flight: 1000/1980...
2026-05-09 05:26:42,332 | INFO |   Pre-flight: 1100/1980...


[src/libmpg123/id3.c:INT123_id3_to_utf8():394] warning: Weird tag size 93 for encoding 1 - I will probably trim too early or something but I think the MP3 is broken.


2026-05-09 05:26:43,627 | INFO |   Pre-flight: 1200/1980...
2026-05-09 05:26:45,061 | INFO |   Pre-flight: 1300/1980...
2026-05-09 05:26:46,615 | INFO |   Pre-flight: 1400/1980...
2026-05-09 05:26:47,927 | INFO |   Pre-flight: 1500/1980...
2026-05-09 05:26:49,085 | INFO |   Pre-flight: 1600/1980...
2026-05-09 05:26:50,338 | INFO |   Pre-flight: 1700/1980...
2026-05-09 05:26:52,119 | INFO |   Pre-flight: 1800/1980...
2026-05-09 05:26:53,799 | INFO |   Pre-flight: 1900/1980...
2026-05-09 05:26:54,633 | INFO | Valid file list cached → checkpoints/deepfake_valid_indices.json
2026-05-09 05:26:54,655 | INFO | Loaded cached file list (1980 valid, 0 skipped).
2026-05-09 05:26:54,666 | INFO | Loaded cached file list (1980 valid, 0 skipped).
2026-05-09 05:26:54,670 | INFO | DFE full : 1,980  (1,167 real / 813 fake)
2026-05-09 05:26:54,671 | INFO | DFE val  : 296  (148 real / 148 fake)  [14.9%]
2026-05-09 05:26:54,672 | INFO | DFE test : 1,684  (1,019 real / 665 fake)  [85.1%]
2026-05-09 05:26:54

[src/libmpg123/parse.c:skip_junk():1260] error: Giving up searching valid MPEG header after 65536 bytes of junk.


2026-05-09 05:27:28,030 | INFO |   dfe_full batch: (32, 64600)  labels=[1, 1, 0, 1]
2026-05-09 05:28:05,353 | INFO |    dfe_val batch: (32, 64600)  labels=[1, 1, 1, 1]


[src/libmpg123/parse.c:skip_junk():1260] error: Giving up searching valid MPEG header after 65536 bytes of junk.


2026-05-09 05:28:32,115 | INFO |   dfe_test batch: (32, 64600)  labels=[1, 1, 0, 1]
2026-05-09 05:28:32,118 | INFO | All loaders confirmed.


In [8]:
## Load Parent Checkpoints
# CORRECT — matches your checkpoint
d_args = {
    'filts'       : [128, [128, 128], [128, 512]],
    'first_conv'  : 1024,
    'in_channels' : 1,
    'gru_node'    : 1024,
    'nb_gru_layer': 3,
    'nb_fc_node'  : 1024,
    'nb_classes'  : 2,
}
logger.info('Loading parent checkpoints...')

def load_parent(path, label):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        sd = ckpt['model_state_dict']
        logger.info(f'{label}: full checkpoint, epoch={ckpt.get("epoch","?")}')
    else:
        sd = ckpt
        logger.info(f'{label}: legacy weights-only checkpoint')
    logger.info(f'{label}: {len(sd)} keys')
    return sd

p1_state = load_parent(MODEL_PATHS['la_parent'],  'LA parent')
p2_state = load_parent(MODEL_PATHS['itw_parent'], 'ITW parent')

# ── Key consistency checks ────────────────────────────────────────────────────
assert set(p1_state.keys()) == set(p2_state.keys()), \
    f'Parent key mismatch!\n  LA only : {set(p1_state.keys()) - set(p2_state.keys())}\n  ITW only: {set(p2_state.keys()) - set(p1_state.keys())}'

# Validate against live v2 model (class is RawNet, not RawNet2)
# CORRECT
_d = {
    'filts'       : [128, [128, 128], [128, 512]],
    'first_conv'  : 1024,
    'in_channels' : 1,
    'gru_node'    : 1024,
    'nb_gru_layer': 3,
    'nb_fc_node'  : 1024,
    'nb_classes'  : 2,
}
_probe = RawNet(_d, DEVICE).to(DEVICE)
_model_keys = set(_probe.state_dict().keys())
del _probe

missing = _model_keys - set(p1_state.keys())
extra   = set(p1_state.keys()) - _model_keys

if missing or extra:
    if missing: logger.warning(f'Missing in checkpoint: {missing}')
    if extra:   logger.warning(f'Extra in checkpoint  : {extra}')
    raise RuntimeError('Parent keys do not match v2 RawNet. Inspect above.')

logger.info('Both parents loaded and keys verified against v2 RawNet.')

2026-05-09 05:31:15,113 | INFO | Loading parent checkpoints...
2026-05-09 05:31:17,553 | INFO | LA parent: full checkpoint, epoch=50
2026-05-09 05:31:17,553 | INFO | LA parent: 119 keys
2026-05-09 05:31:19,005 | INFO | ITW parent: full checkpoint, epoch=59
2026-05-09 05:31:19,005 | INFO | ITW parent: 119 keys
2026-05-09 05:31:19,263 | INFO | Both parents loaded and keys verified against v2 RawNet.


In [9]:
## Core Utilities: Feature Extraction, CKA, Metrics
# ── Core Utilities: Feature Extraction, CKA, Metrics ──────────────────────────

def compute_linear_cka(x: torch.Tensor, y: torch.Tensor) -> float:
    """Linear Centred Kernel Alignment. Always computed on CPU to avoid
    device-mismatch errors when running dual-GPU (cuda:0 vs cuda:1)."""
    x = x.cpu().float()
    y = y.cpu().float()
    x = x - x.mean(0)
    y = y - y.mean(0)
    num   = torch.norm(torch.matmul(x.t(), y)) ** 2
    denom = (torch.norm(torch.matmul(x.t(), x)) *
             torch.norm(torch.matmul(y.t(), y)) + 1e-8)
    return (num / denom).item()


def extract_reference_features(model, loader, device):
    """
    Extract features at 3 depths from parent-1 for CKA reference.
    Returns dict: 'early' (block1), 'mid' (block3), 'final' (GRU last step).
    All tensors shape (N, D), CPU.
    """
    model.eval()
    fe, fm, ff = [], [], []

    def h_early(m, inp, out):
        fe.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_mid(m, inp, out):
        fm.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_gru(m, inp, out):
        seq, _ = out
        ff.append(seq[:, -1, :].detach().cpu())

    hooks = [
        model.block1.register_forward_hook(h_early),
        model.block3.register_forward_hook(h_mid),
        model.gru.register_forward_hook(h_gru),
    ]
    with torch.no_grad():
        for x, _ in tqdm(loader, desc='  Extracting P1 features', leave=False):
            model(x.to(device), is_test=True)
    for h in hooks: h.remove()

    return {
        'early': torch.cat(fe),
        'mid':   torch.cat(fm),
        'final': torch.cat(ff),
    }


def full_metrics(labels, preds, scores):
    """Compute all evaluation metrics given labels, hard preds, and soft scores."""
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    return {
        'accuracy'  : float(accuracy_score(labels, preds)),
        'auc'       : float(roc_auc_score(labels, scores)),
        'bal_acc'   : float(balanced_accuracy_score(labels, preds)),
        'precision' : float(precision_score(labels, preds, zero_division=0)),
        'recall'    : float(recall_score(labels, preds, zero_division=0)),
        'f1'        : float(f1_score(labels, preds, zero_division=0)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'confusion_matrix': cm.tolist(),
    }


def evaluate_candidate(model, loader, device, p1_feats, config):
    """
    Evaluate one candidate model on the validation set.
    Returns (metrics_dict, fitness_score).

    Fitness (MeGA-IA v2 formulation from architecture doc):
        S(c) = AUC - λ1·CKA_multi + λ2·Confidence

    CKA_multi: simple mean across [early, mid, final] layers.
    Confidence: mean(p² + (1-p)²) — confidence bonus (added, not subtracted).
    Entropy penalty (v1 style): mean(-Σ p·log2(p)) — kept for legacy configs.
    """
    model.eval()
    all_scores, all_labels = [], []
    cf_e, cf_m, cf_f = [], [], []

    def h_e(m, inp, out):
        cf_e.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_m(m, inp, out):
        cf_m.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_g(m, inp, out):
        seq, _ = out
        cf_f.append(seq[:, -1, :].detach().cpu())

    hooks = [
        model.block1.register_forward_hook(h_e),
        model.block3.register_forward_hook(h_m),
        model.gru.register_forward_hook(h_g),
    ]
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device), is_test=True)
            all_scores.append(out[:, 1].cpu().numpy())
            all_labels.append(y.numpy())
    for h in hooks: h.remove()

    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels)
    preds  = (scores > 0.5).astype(int)

    feat_e = torch.cat(cf_e)
    feat_m = torch.cat(cf_m)
    feat_f = torch.cat(cf_f)

    met = full_metrics(labels, preds, scores)
    met['scores'] = scores.tolist()
    met['labels'] = labels.tolist()

    # ── Confidence bonus (v2 doc): mean(p² + (1-p)²) ─────────────────────
    p = scores.clip(1e-8, 1 - 1e-8)
    met['confidence'] = float(np.mean(p ** 2 + (1 - p) ** 2))

    # ── Legacy entropy penalty (v1 style, kept for E01-E10 compat) ────────
    p_mat = np.stack([1 - p, p], axis=1)
    met['entropy'] = float(-np.mean(np.sum(p_mat * np.log2(p_mat), axis=1)))

    # ── CKA: mode controlled by config ────────────────────────────────────
    cka_mode = config.get('cka_mode', 'final')
    if cka_mode == 'multilayer':
        cka_e = compute_linear_cka(feat_e, p1_feats['early'])
        cka_m = compute_linear_cka(feat_m, p1_feats['mid'])
        cka_f = compute_linear_cka(feat_f, p1_feats['final'])
        # v2 doc: simple mean across layers (equal weight)
        cka_score = (cka_e + cka_m + cka_f) / 3.0
        met['cka_early'] = float(cka_e)
        met['cka_mid']   = float(cka_m)
        met['cka_final'] = float(cka_f)
    else:
        cka_score = compute_linear_cka(feat_f, p1_feats['final'])
    met['cka'] = float(cka_score)

    # ── Composite fitness ─────────────────────────────────────────────────
    ft_type = config.get('fitness_type', 'accuracy')
    base    = {'accuracy': met['accuracy'],
               'auc':      met['auc'],
               'balanced': met['bal_acc'],
               'auc_f1':   0.6 * met['auc'] + 0.4 * met['f1']}[ft_type]

    penalty_mode = config.get('penalty_mode', 'entropy')   # 'entropy' | 'confidence'
    if penalty_mode == 'confidence':
        # v2 doc: AUC - λ1·CKA + λ2·Confidence  (confidence is a BONUS)
        fitness = base - config['lambda1'] * cka_score + config['lambda2'] * met['confidence']
    else:
        # v1 legacy: AUC - λ1·interference - λ2·entropy
        fitness = base - config['lambda1'] * cka_score - config['lambda2'] * met['entropy']

    met['fitness'] = float(fitness)
    return met, float(fitness)


# ── Checkpoint helpers ─────────────────────────────────────────────────────────
def save_exp_results(ckpt_dir, exp_id, best_val_metrics, history, best_genome):
    out = {
        'exp_id': exp_id,
        'best_val_metrics': {k: v for k, v in best_val_metrics.items()
                             if k not in ('scores', 'labels')},
        'history':     history,
        'best_genome': best_genome.tolist() if hasattr(best_genome, 'tolist') else best_genome,
    }
    path = os.path.join(ckpt_dir, f'{exp_id}_results.json')
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)
    logger.info(f'Results saved → {path}')


def load_exp_results(ckpt_dir, exp_id):
    path = os.path.join(ckpt_dir, f'{exp_id}_results.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def save_test_results(ckpt_dir, exp_id, dataset_key, test_metrics):
    """dataset_key: 'dfe_full' | 'la' | 'itw'"""
    out = {k: v for k, v in test_metrics.items() if k not in ('scores', 'labels')}
    path = os.path.join(ckpt_dir, f'{exp_id}_test_{dataset_key}.json')
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)


def load_test_results(ckpt_dir, exp_id, dataset_key):
    path = os.path.join(ckpt_dir, f'{exp_id}_test_{dataset_key}.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


logger.info('Core utilities defined.')

2026-05-09 05:31:29,921 | INFO | Core utilities defined.


In [10]:
## MeGA-IA Evolutionary Engine (Multi-GPU)
import concurrent.futures

def evaluate_genome_on_device(args):
    """
    Worker function: builds one model on a specific device and evaluates it.
    Designed to run in a ThreadPoolExecutor (GIL is released during CUDA ops).
    """
    genome, p1_state, p2_state, d_args, all_keys, n_fl, alpha_mode, \
        val_loader, p1_feats, config, device = args

    float_keys = [k for k in p1_state if p1_state[k].dtype.is_floating_point]

    def _key_to_group6(k):
        if any(k.startswith(p) for p in ('Sinc_conv', 'first_bn')):
            return 0
        if any(k.startswith(p) for p in ('block0', 'block1', 'fc_attention0', 'fc_attention1')):
            return 1
        if any(k.startswith(p) for p in ('block2', 'block3', 'fc_attention2', 'fc_attention3')):
            return 2
        if any(k.startswith(p) for p in ('block4', 'block5', 'fc_attention4', 'fc_attention5', 'bn_before_gru')):
            return 3
        if k.startswith('gru'):
            return 4
        return 5

    m = RawNet(d_args, device).to(device)
    sd, fi = {}, 0
    for k in all_keys:
        v1 = p1_state[k].to(device)
        v2 = p2_state[k].to(device)
        if not v1.dtype.is_floating_point:
            sd[k] = v1.clone()
        else:
            if alpha_mode == 'global':
                a = float(genome[0])
            elif alpha_mode == 'segment3':
                seg = min(fi * 3 // n_fl, 2)
                a = float(genome[seg])
            else:
                a = float(genome[_key_to_group6(k)])
            sd[k] = a * v1 + (1.0 - a) * v2
            fi += 1
    m.load_state_dict(sd)

    # Rebuild p1_feats on the target device if needed
    p1_feats_dev = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                    for k, v in p1_feats.items()}

    met, fit = evaluate_candidate(m, val_loader, device, p1_feats_dev, config)
    del m
    torch.cuda.empty_cache()
    return met, fit


def run_mega_ia(config, p1_state, p2_state, d_args, device,
                val_loader, p1_feats, checkpoint_dir, device2=None):
    """
    MeGA-IA with dual-GPU parallel genome evaluation.

    If device2 is provided (and differs from device), the population is
    split evenly: even indices → device, odd indices → device2.
    Evaluation is parallelised with ThreadPoolExecutor(max_workers=2).
    """
    if device2 is None:
        device2 = device
    dual_gpu = (str(device) != str(device2))

    exp_id      = config['id']
    evo_ckpt    = os.path.join(checkpoint_dir, f'{exp_id}_evo_state.json')
    weights_pth = os.path.join(checkpoint_dir, f'{exp_id}_best_weights.pth')

    float_keys  = [k for k in p1_state if p1_state[k].dtype.is_floating_point]
    n_fl        = len(float_keys)
    all_keys    = list(p1_state.keys())
    alpha_mode  = config.get('alpha_mode', 'global')

    def _key_to_group6(k):
        if any(k.startswith(p) for p in ('Sinc_conv', 'first_bn')):
            return 0
        if any(k.startswith(p) for p in ('block0', 'block1', 'fc_attention0', 'fc_attention1')):
            return 1
        if any(k.startswith(p) for p in ('block2', 'block3', 'fc_attention2', 'fc_attention3')):
            return 2
        if any(k.startswith(p) for p in ('block4', 'block5', 'fc_attention4', 'fc_attention5', 'bn_before_gru')):
            return 3
        if k.startswith('gru'):
            return 4
        return 5

    genome_size = {'global': 1, 'segment3': 3, 'layerwise6': 6}[alpha_mode]

    def make_model(genome, dev=None):
        if dev is None:
            dev = device
        m = RawNet(d_args, dev).to(dev)
        sd, fi = {}, 0
        for k in all_keys:
            v1 = p1_state[k].to(dev)
            v2 = p2_state[k].to(dev)
            if not v1.dtype.is_floating_point:
                sd[k] = v1.clone()
            else:
                if alpha_mode == 'global':
                    a = float(genome[0])
                elif alpha_mode == 'segment3':
                    seg = min(fi * 3 // n_fl, 2)
                    a = float(genome[seg])
                else:
                    a = float(genome[_key_to_group6(k)])
                sd[k] = a * v1 + (1.0 - a) * v2
                fi += 1
        m.load_state_dict(sd)
        return m

    def rnd_genome():
        return np.random.uniform(0.0, 1.0, genome_size)

    def crossover(g1, g2):
        mode = config.get('crossover_mode', 'blend')
        if mode == 'blend':
            beta = np.random.rand()
            return beta * g1 + (1.0 - beta) * g2
        elif mode == 'one_point':
            if genome_size <= 1:
                return g1.copy() if np.random.rand() < 0.5 else g2.copy()
            cut = np.random.randint(1, genome_size)
            return np.concatenate([g1[:cut], g2[cut:]])
        else:
            mask = np.random.rand(genome_size) < 0.5
            return np.where(mask, g1, g2)

    def mutate(genome, gen):
        decay = math.exp(-0.5 * gen / max(config['generations'], 1))
        sigma = config.get('sigma_base', 0.10) * decay
        return np.clip(genome + np.random.normal(0, sigma, genome_size), 0.0, 1.0)

    def tournament(pop, fits, k=4):
        idx = np.random.choice(len(pop), min(k, len(pop)), replace=False)
        best_i = idx[np.argmax([fits[i] for i in idx])]
        return pop[best_i].copy()

    # ── Init / resume ─────────────────────────────────────────────────────
    start_gen    = 0
    population   = [rnd_genome() for _ in range(config['pop_size'])]
    best_genome  = population[0].copy()
    best_fitness = -np.inf
    best_val_met = None
    stagnation   = 0
    hist_keys    = ['best_fit', 'mean_fit', 'worst_fit', 'best_auc', 'best_bal',
                    'best_f1', 'mean_auc', 'diversity', 'best_prec', 'best_rec',
                    'best_cka', 'best_conf']
    history = {k: [] for k in hist_keys}

    if os.path.exists(evo_ckpt):
        try:
            with open(evo_ckpt) as f:
                ec = json.load(f)
            start_gen    = ec['last_gen'] + 1
            population   = [np.array(g) for g in ec['population']]
            best_genome  = np.array(ec['best_genome'])
            best_fitness = ec['best_fitness']
            best_val_met = ec.get('best_val_met')
            stagnation   = ec.get('stagnation', 0)
            history      = ec['history']
            logger.info(f'  ♻️  [{exp_id}] Resumed from gen {start_gen}')
        except Exception as e:
            logger.warning(f'  Evo checkpoint corrupt ({e}) — starting fresh')

    total_gens = config['generations']
    pop_sz     = config['pop_size']

    # ── Evolutionary loop ─────────────────────────────────────────────────
    for gen in range(start_gen, total_gens):
        t0 = time.time()
        logger.info(f'')
        logger.info(f'┌─ [{config["name"]}]  Gen {gen+1:2d}/{total_gens}'
                    + (f'  [dual-GPU: {device} + {device2}]' if dual_gpu else ''))

        gen_fits, gen_mets = [None] * pop_sz, [None] * pop_sz

        if dual_gpu:
            # ── Parallel evaluation across 2 GPUs ─────────────────────────
            # Build per-genome args; alternate devices across the population.
            eval_args = []
            for ci, genome in enumerate(population):
                dev = device if ci % 2 == 0 else device2
                eval_args.append((
                    genome, p1_state, p2_state, d_args, all_keys, n_fl,
                    alpha_mode, val_loader, p1_feats, config, dev
                ))

            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
                futures = {pool.submit(evaluate_genome_on_device, args): ci
                           for ci, args in enumerate(eval_args)}
                for fut in concurrent.futures.as_completed(futures):
                    ci = futures[fut]
                    met, fit = fut.result()
                    gen_fits[ci] = fit
                    gen_mets[ci] = met
                    dev_lbl = f'cuda:{ci % 2}'
                    cka_str = (
                        f'cka_e={met["cka_early"]:.3f} cka_m={met["cka_mid"]:.3f} cka_f={met["cka_final"]:.3f}'
                        if 'cka_early' in met else f'cka={met["cka"]:.4f}'
                    )
                    logger.info(
                        f'│ [{ci+1:2d}/{pop_sz}][{dev_lbl}] '
                        f'fit={fit:+.4f}  AUC={met["auc"]:.4f}  '
                        f'bal={met["bal_acc"]:.4f}  F1={met["f1"]:.4f}  '
                        f'rec={met["recall"]:.4f}  prec={met["precision"]:.4f}  '
                        f'{cka_str}  conf={met["confidence"]:.4f}'
                    )
        else:
            # ── Single-GPU fallback (original sequential loop) ────────────
            for ci, genome in enumerate(population):
                model = make_model(genome)
                met, fit = evaluate_candidate(model, val_loader, device, p1_feats, config)
                gen_fits[ci] = fit
                gen_mets[ci] = met
                del model
                torch.cuda.empty_cache()
                cka_str = (
                    f'cka_e={met["cka_early"]:.3f} cka_m={met["cka_mid"]:.3f} cka_f={met["cka_final"]:.3f}'
                    if 'cka_early' in met else f'cka={met["cka"]:.4f}'
                )
                logger.info(
                    f'│ [{ci+1:2d}/{pop_sz}] fit={fit:+.4f}  AUC={met["auc"]:.4f}  '
                    f'bal={met["bal_acc"]:.4f}  F1={met["f1"]:.4f}  '
                    f'rec={met["recall"]:.4f}  prec={met["precision"]:.4f}  '
                    f'{cka_str}  conf={met["confidence"]:.4f}'
                )

        # ── Generation summary (unchanged) ────────────────────────────────
        gen_best_i = int(np.argmax(gen_fits))
        gen_best_f = gen_fits[gen_best_i]
        gen_best_m = gen_mets[gen_best_i]
        mean_fit   = float(np.mean(gen_fits))
        worst_fit  = float(np.min(gen_fits))
        mean_auc   = float(np.mean([m['auc'] for m in gen_mets]))
        diversity  = float(np.std(np.vstack(population)))

        improved = gen_best_f > best_fitness
        if improved:
            best_fitness = gen_best_f
            best_genome  = population[gen_best_i].copy()
            best_val_met = {k: v for k, v in gen_best_m.items()
                            if k not in ('scores', 'labels')}
            bm = make_model(best_genome)
            torch.save(bm.state_dict(), weights_pth)
            del bm
            torch.cuda.empty_cache()
            stagnation = 0
        else:
            stagnation += 1

        history['best_fit'].append(float(gen_best_f))
        history['mean_fit'].append(mean_fit)
        history['worst_fit'].append(worst_fit)
        history['best_auc'].append(float(gen_best_m['auc']))
        history['best_bal'].append(float(gen_best_m['bal_acc']))
        history['best_f1'].append(float(gen_best_m['f1']))
        history['mean_auc'].append(mean_auc)
        history['diversity'].append(diversity)
        history['best_prec'].append(float(gen_best_m['precision']))
        history['best_rec'].append(float(gen_best_m['recall']))
        history['best_cka'].append(float(gen_best_m['cka']))
        history['best_conf'].append(float(gen_best_m['confidence']))

        star = ' ⭐ NEW BEST' if improved else ''
        logger.info(f'├─ Gen summary  best={gen_best_f:.4f}  mean={mean_fit:.4f}  '
                    f'worst={worst_fit:.4f}  div={diversity:.4f}  ({time.time()-t0:.0f}s){star}')
        logger.info(f'│  Gen best:  AUC={gen_best_m["auc"]:.4f}  '
                    f'bal={gen_best_m["bal_acc"]:.4f}  F1={gen_best_m["f1"]:.4f}  '
                    f'conf={gen_best_m["confidence"]:.4f}')
        logger.info(f'│  Global:    fit={best_fitness:.4f}  stagnation={stagnation} gen(s)')
        if best_genome is not None:
            logger.info(f'│  Genome:    {[f"{v:.3f}" for v in best_genome]}')

        if (config.get('diversity_injection') and
                stagnation >= config.get('diversity_patience', 5)):
            n_inj   = max(1, pop_sz // 4)
            worst_i = np.argsort(gen_fits)[:n_inj]
            for wi in worst_i:
                population[wi] = rnd_genome()
            logger.info(f'│  💉 Diversity injection: replaced {n_inj} worst (stagnation reset)')
            stagnation = 0

        new_pop = [best_genome.copy()]
        while len(new_pop) < pop_sz:
            pa    = tournament(population, gen_fits)
            pb    = tournament(population, gen_fits)
            child = crossover(pa, pb)
            if np.random.rand() < 0.20:
                child = mutate(child, gen)
            new_pop.append(child)
        population = new_pop

        ec_out = {
            'last_gen': gen, 'population': [g.tolist() for g in population],
            'best_genome': best_genome.tolist(), 'best_fitness': float(best_fitness),
            'best_val_met': best_val_met, 'stagnation': stagnation, 'history': history,
        }
        with open(evo_ckpt, 'w') as f:
            json.dump(ec_out, f)
        logger.info(f'└─ Gen {gen+1} complete. Checkpoint saved.')

    logger.info(f'')
    logger.info(f'🏆 [{config["name"]}] Evolution finished!')
    logger.info(f'   Best fitness : {best_fitness:.4f}')
    logger.info(f'   Best val AUC : {best_val_met["auc"]:.4f}')
    logger.info(f'   Best val bal : {best_val_met["bal_acc"]:.4f}')
    logger.info(f'   Best val F1  : {best_val_met["f1"]:.4f}')
    return best_val_met, history, best_genome


logger.info('MeGA-IA engine defined (multi-GPU).')

2026-05-09 05:31:39,321 | INFO | MeGA-IA engine defined (multi-GPU).


In [ ]:
## Experiment Configurations
# ── Experiment Configurations ──────────────────────────────────────────────────
# v2 experiments start from E01.
# E01 = MeGA-IA v2 Full (AUC + raised penalties + 3-seg alpha + multilayer CKA
#        + confidence bonus + blend crossover) — the deduced best config from v1.
#
# ─────────────────────────────────────────────────────────────────────────────
# NEW EXPERIMENTS (E11–E17) added below the original list.
# None of the original E01 config has been modified.
# ─────────────────────────────────────────────────────────────────────────────

EXPERIMENTS = [

    # ── E00: Exact doc config (MeGA-IA v2 as specified in .docx) ──────────
    # Formula: S(c) = AUC - λ1·CKA_multi + λ2·H  (H = p²+(1-p)², λ1=0.30, λ2=0.15)
    # 3-segment alpha | blend crossover | σ=0.10 | multilayer CKA
    # Fitness on DFE-val 15%  |  Test on DFE-test 85% (no overlap)
   dict(id='E00_mega_ia_doc',  name='E00 MeGA-IA Doc Config',
     group='v2 — Doc Baseline',
     fitness_type='auc',      lambda1=0.30, lambda2=0.15,
     penalty_mode='entropy',  crossover_mode='blend',
     generations=10, pop_size=15, alpha_mode='segment3', cka_mode='multilayer',
     diversity_injection=True,  diversity_patience=5, sigma_base=0.10,
     dfe_val_fitness=True,
     description='Doc config: AUC - λ1·CKA_multi - λ2·entropy | 3-seg α | DFE-val 15% fitness | DFE-test 85% eval'),

    # ── v2 Experiments ────────────────────────────────────────────────────
    dict(id='E01_mega_ia_v2',   name='E01 MeGA-IA v2 (Full)',
         group='v2 — Integrated',
         fitness_type='auc',      lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='blend',
         generations=10, pop_size=15, alpha_mode='segment3', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=5, sigma_base=0.10,
         description='AUC - λ1·CKA_multi + λ2·Confidence | 3-seg α | blend crossover | deduced best from v1'),

    # ── E11: Layerwise 6-gene genome + one-point crossover ────────────────
    # Tests finer-grained layer control vs the 3-segment genome.
    # Each of the 6 architecture groups (sinc+bn, blk01, blk23, blk45, gru, fc)
    # gets its own independent α.  One-point crossover preserves contiguous
    # layer blocks (biologically motivated: adjacent layers are co-adapted).
    dict(id='E11_layerwise6',  name='E11 Layerwise 6-gene',
         group='v3 — Genome',
         fitness_type='auc',      lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='one_point',
         generations=15, pop_size=20, alpha_mode='layerwise6', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=4, sigma_base=0.08,
         description='6-gene genome (sinc+bn/blk01/blk23/blk45/gru/fc) | one-point crossover'),

    # ── E12: AUC + F1 joint fitness (0.6·AUC + 0.4·F1) ───────────────────
    # Rewards both ranking (AUC) and threshold-level detection quality (F1).
    # Particularly useful on class-imbalanced test sets like Deepfake-Evals.
    dict(id='E12_auc_f1',      name='E12 AUC+F1 Fitness',
         group='v3 — Fitness',
         fitness_type='auc_f1',   lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='blend',
         generations=15, pop_size=20, alpha_mode='segment3', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=4, sigma_base=0.10,
         description='Joint fitness: 0.6·AUC + 0.4·F1 − λ1·CKA + λ2·Conf | 3-seg α'),

    # ── E13: DFE-val as fitness set; DFE-test as held-out evaluation ───────
    # Directly addresses the data-leakage concern noted in the final cell of
    # the original notebook.  Fitness is computed on dfe_val_loader (15% DFE),
    # and the final model is tested on dfe_test_loader (85% DFE, no overlap).
    # The run loop below detects the 'dfe_val_fitness' flag and swaps loaders.
    dict(id='E13_dfe_val_fitness', name='E13 DFE-val Fitness',
         group='v3 — Val split',
         fitness_type='auc',      lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='blend',
         generations=10, pop_size=15, alpha_mode='segment3', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=5, sigma_base=0.10,
         dfe_val_fitness=True,   # ← flag: use dfe_val_loader for fitness
         description='v2-E01 config | fitness on DFE-15% val | test on DFE-85% (no overlap)'),

    # ── E14: AUC+F1 joint fitness + layerwise6 genome (combined best) ─────
    # Combines the two strongest new dimensions: finer genome control (E11)
    # and joint fitness (E12), to see if they compound or cancel.
    dict(id='E14_lw6_auc_f1',  name='E14 Layerwise6 + AUC+F1',
         group='v3 — Combined',
         fitness_type='auc_f1',   lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='one_point',
         generations=15, pop_size=20, alpha_mode='layerwise6', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=4, sigma_base=0.08,
         description='6-gene genome + joint AUC+F1 fitness + one-point crossover'),

    # ── E15: Adapter fine-tuning post genome selection ─────────────────────
    # After the E01 best genome is found, a small 2-layer MLP adapter
    # (hidden=64) is appended after the GRU output and fine-tuned for 3 epochs
    # on dfe_val_loader.  The frozen backbone provides a strong representation;
    # the adapter learns a task-specific projection.  Breaks the hard convex-
    # hull constraint of interpolation.
    # NOTE: this experiment runs the adapter fine-tuning loop defined in
    # the 'NEW CELL: AdaptedRawNet' cell below, not run_mega_ia.
    # The run loop handles it via the 'adapter_finetune' flag.
    dict(id='E15_adapter',     name='E15 Adapter Fine-tuning',
         group='v3 — Architecture',
         fitness_type='auc',      lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='blend',
         generations=10, pop_size=15, alpha_mode='segment3', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=5, sigma_base=0.10,
         adapter_finetune=True, adapter_dim=64, adapter_epochs=3, adapter_lr=1e-3,
         description='E01 genome search + 2-layer MLP adapter (hidden=64) fine-tuned 3 epochs on DFE-val'),

    # ── E16: Temperature calibration post selection ────────────────────────
    # After E01 genome selection, learns a scalar temperature T by minimising
    # NLL on dfe_val_loader, then re-scores dfe_test_loader with softmax(logits/T).
    # Reports AUC and ECE before + after calibration.
    # Uses the 'temp_calibration' flag; handled in run loop below.
    dict(id='E16_temp_calib',  name='E16 Temperature Calibration',
         group='v3 — Calibration',
         fitness_type='auc',      lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='blend',
         generations=10, pop_size=15, alpha_mode='segment3', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=5, sigma_base=0.10,
         temp_calibration=True,
         description='E01 genome search + temperature scaling calibrated on DFE-val'),

    # ── E17: Island model — 3 sub-populations, different fitness foci ──────
    # 3 independent islands of 8 genomes each (total=24 evals/gen, ≈ pop=24).
    #   Island A: fitness_type='auc'
    #   Island B: fitness_type='auc_f1'
    #   Island C: fitness_type='balanced'
    # Every 5 gens the best genome from each island migrates to the next.
    # Final candidate = global best across all islands.
    # Uses the 'island_model' flag; handled by run_island_model() below.
    dict(id='E17_islands',     name='E17 Island Model',
         group='v3 — Diversity',
         fitness_type='auc',      lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='blend',
         generations=15, pop_size=8,  alpha_mode='segment3', cka_mode='multilayer',
         diversity_injection=False, diversity_patience=5, sigma_base=0.10,
         island_model=True, n_islands=3, migration_every=5,
         description='3 islands (AUC/AUC+F1/balanced) × pop=8 | migration every 5 gens'),
]

# ── Overview table ────────────────────────────────────────────────────────────
print(f'{"ID":<26} {"Group":<22} {"Fit":<10} {"λ1":<5} {"λ2":<5} '
      f'{"G":<4} {"P":<4} {"Alpha":<12} {"CKA":<12} {"Pen":<12} {"Status"}')
print('─' * 130)
for e in EXPERIMENTS:
    status = '✅ done' if load_exp_results(str(CKPT_DIR), e['id']) else ''
    print(f'{e["id"]:<26} {e["group"]:<22} {e["fitness_type"]:<10} '
          f'{e["lambda1"]:<5} {e["lambda2"]:<5} {e["generations"]:<4} '
          f'{e["pop_size"]:<4} {e["alpha_mode"]:<12} {e["cka_mode"]:<12} '
          f'{e.get("penalty_mode","entropy"):<12} {status}')

In [ ]:
## Pre-extract Parent 1 Reference Features
# ── Pre-Extract Parent 1 Reference Features ────────────────────────────────────
# Runs once on the full val_loader. Reused by all experiments.

logger.info('Extracting Parent 1 reference features on full val set...')
logger.info('(This runs once — reused by all experiments)')

p1_model = RawNet(d_args, DEVICE).to(DEVICE)
p1_model.load_state_dict(p1_state)
p1_model.eval()

p1_feats = extract_reference_features(p1_model, val_loader, DEVICE)
del p1_model
if str(DEVICE) == 'cuda': torch.cuda.empty_cache()

for key, tensor in p1_feats.items():
    logger.info(f'  {key:6s}: {tuple(tensor.shape)}  dtype={tensor.dtype}')

logger.info('Parent 1 features ready.')

# ── Also extract P1 features on DFE-val (needed by E13/E15/E16) ───────────────
logger.info('Extracting Parent 1 reference features on DFE-val set...')
p1_model_dfe = RawNet(d_args, DEVICE).to(DEVICE)
p1_model_dfe.load_state_dict(p1_state)
p1_model_dfe.eval()

p1_feats_dfe = extract_reference_features(p1_model_dfe, dfe_val_loader, DEVICE)
del p1_model_dfe
if str(DEVICE) == 'cuda': torch.cuda.empty_cache()

for key, tensor in p1_feats_dfe.items():
    logger.info(f'  DFE {key:6s}: {tuple(tensor.shape)}  dtype={tensor.dtype}')

logger.info('Parent 1 DFE-val features ready.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# NEW CELL: Extended Architecture Components
# Adds AdaptedRawNet, temperature calibration, and island model runner.
# None of the cells above are modified.
# ════════════════════════════════════════════════════════════════════════════════

# ── AdaptedRawNet: frozen interpolated backbone + small trainable adapter ──────
# Used by E15.  A 2-layer MLP (hidden_dim=adapter_dim) is appended after the
# GRU output of any RawNet model.  The backbone is frozen; only the adapter
# is trained.  This lets the model reach points outside the P1–P2 convex hull.

class AdaptedRawNet(nn.Module):
    """
    Wraps a frozen interpolated RawNet with a small trainable MLP adapter.

    Forward path:
        raw waveform
        → frozen RawNet backbone (up to GRU last step, 1024-d)
        → adapter: Linear(1024→adapter_dim) → GELU → Linear(adapter_dim→2)
        → logits  (or softmax when is_test=True)

    The original fc1_gru / fc2_gru are bypassed — the adapter replaces them.
    This is achieved by hooking the GRU output instead of running full forward.
    """

    def __init__(self, base_model: RawNet, adapter_dim: int = 64):
        super().__init__()

        # ── Freeze backbone ───────────────────────────────────────────────
        self.backbone = base_model
        for p in self.backbone.parameters():
            p.requires_grad_(False)

        # ── Trainable adapter head ────────────────────────────────────────
        gru_hidden = base_model.gru.hidden_size  # 1024
        self.adapter = nn.Sequential(
            nn.Linear(gru_hidden, adapter_dim),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(adapter_dim, 2),
        )
        logger.info(f'AdaptedRawNet: backbone frozen ({sum(p.numel() for p in self.backbone.parameters()):,} params) | '
                    f'adapter trainable ({sum(p.numel() for p in self.adapter.parameters()):,} params)')

    def _gru_features(self, x):
        """Run backbone up to and including GRU; return last-step hidden (B, H)."""
        nb_samp  = x.shape[0]
        len_seq  = x.shape[1]
        x = x.view(nb_samp, 1, len_seq)

        bk = self.backbone
        x  = bk.Sinc_conv(x)
        x  = F.max_pool1d(torch.abs(x), 3)
        x  = bk.first_bn(x)
        x  = bk.selu(x)

        sig = bk.sig
        ap  = bk.avgpool

        for blk, att in [
            (bk.block0, bk.fc_attention0),
            (bk.block1, bk.fc_attention1),
            (bk.block2, bk.fc_attention2),
            (bk.block3, bk.fc_attention3),
            (bk.block4, bk.fc_attention4),
            (bk.block5, bk.fc_attention5),
        ]:
            xi = blk(x)
            yi = ap(xi).view(xi.size(0), -1)
            yi = att(yi)
            yi = sig(yi).view(yi.size(0), yi.size(1), -1)
            x  = xi * yi + yi

        x = bk.bn_before_gru(x)
        x = bk.selu(x)
        x = x.permute(0, 2, 1)
        bk.gru.flatten_parameters()
        x, _ = bk.gru(x)
        return x[:, -1, :]   # (B, 1024)

    def forward(self, x, is_test=False):
        feats  = self._gru_features(x)     # (B, 1024) — no grad through backbone
        logits = self.adapter(feats)        # (B, 2)    — grad through adapter
        return F.softmax(logits, dim=1) if is_test else logits


def finetune_adapter(adapted_model, train_loader, device, epochs=3, lr=1e-3):
    """
    Fine-tune only the adapter head of an AdaptedRawNet.
    Uses cross-entropy loss.  Returns the model (in-place).
    """
    opt   = torch.optim.Adam(adapted_model.adapter.parameters(), lr=lr)
    ce    = nn.CrossEntropyLoss()
    adapted_model.train()
    adapted_model.backbone.eval()  # keep BN in eval mode for frozen backbone

    for ep in range(epochs):
        total_loss, n_correct, n_total = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc=f'  Adapter epoch {ep+1}/{epochs}', leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = adapted_model(x, is_test=False)
            loss   = ce(logits, y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(y)
            n_correct  += (logits.argmax(1) == y).sum().item()
            n_total    += len(y)
        logger.info(f'  Adapter ep {ep+1}: loss={total_loss/n_total:.4f}  acc={n_correct/n_total:.4f}')

    adapted_model.eval()
    return adapted_model


# ── Temperature calibration ────────────────────────────────────────────────────
# Learns a scalar T on a calibration loader (dfe_val_loader) by minimising NLL.
# Re-scores the test set with softmax(logits / T).  Also computes ECE.

from scipy.optimize import minimize_scalar

def collect_logits(model, loader, device):
    """Collect raw logits (before softmax) from a RawNet model."""
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            # Use is_test=False to get raw logits
            logits = model(x.to(device), is_test=False)
            all_logits.append(logits.cpu())
            all_labels.append(y)
    return torch.cat(all_logits), torch.cat(all_labels)


def find_temperature(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """
    Find scalar temperature T that minimises NLL on (logits, labels).
    logits: (N, 2) raw logits; labels: (N,) int.
    Returns optimal T (float).
    """
    def nll(T):
        scaled_probs = F.softmax(logits / T, dim=1)
        # Gather probability of the true class
        true_probs = scaled_probs[range(len(labels)), labels].clamp(1e-8)
        return -true_probs.log().mean().item()

    res = minimize_scalar(nll, bounds=(0.05, 10.0), method='bounded')
    return float(res.x)


def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """
    Expected Calibration Error (ECE) for binary classifier.
    probs: predicted P(class=1), labels: {0,1}.
    """
    bins    = np.linspace(0, 1, n_bins + 1)
    ece     = 0.0
    n_total = len(labels)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        acc  = labels[mask].mean()
        conf = probs[mask].mean()
        ece += (mask.sum() / n_total) * abs(acc - conf)
    return float(ece)


def calibrate_and_evaluate(model, cal_loader, test_loader, device, exp_id, ckpt_dir):
    """
    1. Collect logits on cal_loader → find optimal T.
    2. Collect logits on test_loader → apply T → compute metrics + ECE.
    Returns dict with before/after metrics.
    """
    logger.info('  Collecting calibration logits...')
    cal_logits, cal_labels = collect_logits(model, cal_loader, device)

    T = find_temperature(cal_logits, cal_labels)
    logger.info(f'  Optimal temperature T = {T:.4f}')

    logger.info('  Collecting test logits...')
    tst_logits, tst_labels = collect_logits(model, test_loader, device)
    lbl_np = tst_labels.numpy()

    # Before calibration
    p_before = F.softmax(tst_logits, dim=1)[:, 1].numpy()
    pred_before = (p_before > 0.5).astype(int)
    met_before  = full_metrics(lbl_np, pred_before, p_before)
    met_before['ece']   = compute_ece(p_before, lbl_np)
    met_before['temp']  = 1.0

    # After calibration
    p_after  = F.softmax(tst_logits / T, dim=1)[:, 1].numpy()
    pred_after = (p_after > 0.5).astype(int)
    met_after  = full_metrics(lbl_np, pred_after, p_after)
    met_after['ece']   = compute_ece(p_after, lbl_np)
    met_after['temp']  = T

    calib_results = {'before': met_before, 'after': met_after}
    out_path = os.path.join(ckpt_dir, f'{exp_id}_calibration.json')
    safe = {k: {ck: cv for ck, cv in v.items() if ck not in ('scores','labels')}
            for k, v in calib_results.items()}
    with open(out_path, 'w') as f:
        json.dump(safe, f, indent=2)

    logger.info(f'  Before calib: AUC={met_before["auc"]:.4f}  ECE={met_before["ece"]:.4f}')
    logger.info(f'  After  calib: AUC={met_after["auc"]:.4f}  ECE={met_after["ece"]:.4f}  T={T:.3f}')
    logger.info(f'  Calibration results saved → {out_path}')
    return calib_results


# ── Island model runner ────────────────────────────────────────────────────────
# 3 independent sub-populations, each with a different fitness focus.
# Every migration_every generations the best genome from each island
# is injected into the next island's population.

ISLAND_FITNESS_TYPES = ['auc', 'auc_f1', 'balanced']


def run_island_model(config, p1_state, p2_state, d_args, device,
                     val_loader, p1_feats, checkpoint_dir):
    """
    Island model EA: n_islands sub-populations × pop_size genomes each.
    Migration every migration_every gens (ring topology: island i → island i+1).
    Returns (best_val_metrics, history, best_genome) same as run_mega_ia.
    """
    exp_id         = config['id']
    n_islands      = config.get('n_islands', 3)
    migration_every= config.get('migration_every', 5)
    total_gens     = config['generations']
    pop_sz         = config['pop_size']
    weights_pth    = os.path.join(checkpoint_dir, f'{exp_id}_best_weights.pth')

    # ── Genome helpers (identical to run_mega_ia) ─────────────────────────
    float_keys = [k for k in p1_state if p1_state[k].dtype.is_floating_point]
    n_fl       = len(float_keys)
    all_keys   = list(p1_state.keys())
    genome_size = 3  # island model uses segment3

    def make_model(genome):
        m  = RawNet(d_args, device).to(device)
        sd, fi = {}, 0
        for k in all_keys:
            v1 = p1_state[k].to(device)
            v2 = p2_state[k].to(device)
            if not v1.dtype.is_floating_point:
                sd[k] = v1.clone()
            else:
                seg = min(fi * 3 // n_fl, 2)
                a   = float(genome[seg])
                sd[k] = a * v1 + (1.0 - a) * v2
                fi += 1
        m.load_state_dict(sd)
        return m

    def rnd_genome():
        return np.random.uniform(0.0, 1.0, genome_size)

    def crossover(g1, g2):
        beta = np.random.rand()
        return beta * g1 + (1.0 - beta) * g2

    def mutate(genome, gen):
        decay = math.exp(-0.5 * gen / max(total_gens, 1))
        sigma = config.get('sigma_base', 0.10) * decay
        return np.clip(genome + np.random.normal(0, sigma, genome_size), 0.0, 1.0)

    def tournament(pop, fits, k=4):
        idx    = np.random.choice(len(pop), min(k, len(pop)), replace=False)
        best_i = idx[np.argmax([fits[i] for i in idx])]
        return pop[best_i].copy()

    # ── Initialise islands ────────────────────────────────────────────────
    islands = []
    for isl_i in range(n_islands):
        isl_cfg = dict(config)
        isl_cfg['fitness_type'] = ISLAND_FITNESS_TYPES[isl_i % len(ISLAND_FITNESS_TYPES)]
        islands.append({
            'cfg':      isl_cfg,
            'pop':      [rnd_genome() for _ in range(pop_sz)],
            'fits':     [-np.inf] * pop_sz,
            'best_g':   None,
            'best_f':   -np.inf,
            'best_m':   None,
        })
        logger.info(f'  Island {isl_i}: fitness_type={isl_cfg["fitness_type"]}')

    global_best_f  = -np.inf
    global_best_g  = None
    global_best_m  = None
    hist_keys      = ['best_fit', 'mean_fit', 'best_auc', 'diversity']
    history        = {k: [] for k in hist_keys}

    # ── Main loop ─────────────────────────────────────────────────────────
    for gen in range(total_gens):
        logger.info(f'')
        logger.info(f'┌─ [{config["name"]}] Island Gen {gen+1:2d}/{total_gens}')

        gen_all_fits = []

        for isl_i, isl in enumerate(islands):
            isl_cfg = isl['cfg']
            fits, mets = [], []
            for genome in isl['pop']:
                model = make_model(genome)
                met, fit = evaluate_candidate(model, val_loader, device, p1_feats, isl_cfg)
                fits.append(fit)
                mets.append(met)
                del model
                if str(device) == 'cuda': torch.cuda.empty_cache()

            isl['fits'] = fits
            best_i = int(np.argmax(fits))
            if fits[best_i] > isl['best_f']:
                isl['best_f'] = fits[best_i]
                isl['best_g'] = isl['pop'][best_i].copy()
                isl['best_m'] = {k: v for k, v in mets[best_i].items()
                                 if k not in ('scores', 'labels')}

            # Update global best using AUC as the universal comparison metric
            if mets[best_i]['auc'] > global_best_f:
                global_best_f = mets[best_i]['auc']
                global_best_g = isl['pop'][best_i].copy()
                global_best_m = {k: v for k, v in mets[best_i].items()
                                 if k not in ('scores', 'labels')}
                bm = make_model(global_best_g)
                torch.save(bm.state_dict(), weights_pth)
                del bm
                if str(device) == 'cuda': torch.cuda.empty_cache()

            gen_all_fits.extend(fits)
            logger.info(f'│  Island {isl_i} (fit={isl_cfg["fitness_type"]}): '
                        f'best_fit={fits[best_i]:.4f}  AUC={mets[best_i]["auc"]:.4f}')

            # ── Reproduce within island ───────────────────────────────────
            new_pop = [isl['pop'][best_i].copy()]  # elitism
            while len(new_pop) < pop_sz:
                pa    = tournament(isl['pop'], fits)
                pb    = tournament(isl['pop'], fits)
                child = crossover(pa, pb)
                if np.random.rand() < 0.20:
                    child = mutate(child, gen)
                new_pop.append(child)
            isl['pop'] = new_pop

        # ── Migration (ring: island i → island i+1) ───────────────────────
        if (gen + 1) % migration_every == 0:
            logger.info('│  🔀 Migration step')
            for isl_i in range(n_islands):
                donor_g = islands[isl_i]['best_g']
                if donor_g is not None:
                    recv_i  = (isl_i + 1) % n_islands
                    # Replace the weakest in the receiving island
                    worst_j = int(np.argmin(islands[recv_i]['fits']))
                    islands[recv_i]['pop'][worst_j] = donor_g.copy()
                    logger.info(f'│    Island {isl_i} → Island {recv_i}: '
                                f'genome={[f"{v:.3f}" for v in donor_g]}')

        history['best_fit'].append(float(global_best_f))
        history['mean_fit'].append(float(np.mean(gen_all_fits)))
        history['best_auc'].append(float(global_best_m['auc']) if global_best_m else 0.0)
        history['diversity'].append(float(np.std(np.vstack(
            [g for isl in islands for g in isl['pop']]))))

        logger.info(f'└─ Gen {gen+1} | global_best_AUC={global_best_f:.4f}')

    logger.info(f'🏆 [{config["name"]}] Island evolution finished!')
    logger.info(f'   Global best AUC : {global_best_f:.4f}')
    return global_best_m, history, global_best_g


logger.info('Extended architecture components defined (AdaptedRawNet, calibration, island model).')

In [ ]:
## Run All Experiments
completed_results = {}

for cfg in EXPERIMENTS:
    eid = cfg['id']
    sep = '═' * 70

    existing = load_exp_results(str(CKPT_DIR), eid)
    if existing:
        completed_results[eid] = existing
        logger.info(f'{sep}')
        logger.info(f'[{cfg["name"]}] Already completed — loaded from disk.')
        logger.info(f'  Best val AUC={existing["best_val_metrics"]["auc"]:.4f}  '
                    f'bal={existing["best_val_metrics"]["bal_acc"]:.4f}  '
                    f'F1={existing["best_val_metrics"]["f1"]:.4f}')
        continue

    logger.info(f'{sep}')
    logger.info(f'Starting [{cfg["name"]}]')
    logger.info(f'  {cfg["description"]}')
    logger.info(f'  fitness={cfg["fitness_type"]}  λ1={cfg["lambda1"]}  λ2={cfg["lambda2"]}  '
                f'penalty={cfg.get("penalty_mode","entropy")}')
    logger.info(f'  gens={cfg["generations"]}  pop={cfg["pop_size"]}  '
                f'alpha={cfg["alpha_mode"]}  cka={cfg["cka_mode"]}  '
                f'crossover={cfg.get("crossover_mode","blend")}')
    logger.info(f'  diversity_injection={cfg.get("diversity_injection", False)}')

    exp_start = time.time()

    if cfg.get('dfe_val_fitness', False):
        fitness_loader = dfe_val_loader
        fitness_feats  = p1_feats_dfe
        logger.info('  Using DFE-val (15%) for fitness evaluation.')
    else:
        fitness_loader = val_loader
        fitness_feats  = p1_feats

    if cfg.get('island_model', False):
        best_val_met, history, best_genome = run_island_model(
            cfg, p1_state, p2_state, d_args, DEVICE,
            fitness_loader, fitness_feats, str(CKPT_DIR)
        )
    else:
        best_val_met, history, best_genome = run_mega_ia(
            cfg, p1_state, p2_state, d_args, DEVICE,
            fitness_loader, fitness_feats, str(CKPT_DIR),
            device2=DEVICE2   # ← the only change
        )

    # E15 adapter fine-tuning
    if cfg.get('adapter_finetune', False):
        adapter_wpath = os.path.join(str(CKPT_DIR), f'{eid}_adapter_weights.pth')
        logger.info(f'  Running adapter fine-tuning (E15) on DFE-val...')
        base_model = RawNet(d_args, DEVICE).to(DEVICE)
        base_wpath = os.path.join(str(CKPT_DIR), f'{eid}_best_weights.pth')
        base_model.load_state_dict(torch.load(base_wpath, map_location=DEVICE, weights_only=True))
        adapted = AdaptedRawNet(base_model, adapter_dim=cfg.get('adapter_dim', 64)).to(DEVICE)
        adapted = finetune_adapter(
            adapted, dfe_val_loader, DEVICE,
            epochs=cfg.get('adapter_epochs', 3),
            lr=cfg.get('adapter_lr', 1e-3)
        )
        torch.save(adapted.state_dict(), adapter_wpath)
        logger.info(f'  Adapter weights saved → {adapter_wpath}')
        adapted.eval()
        adp_scores, adp_labels = [], []
        with torch.no_grad():
            for x, y in tqdm(dfe_test_loader, desc='  Adapted model / DFE-test', leave=False):
                out = adapted(x.to(DEVICE), is_test=True)
                adp_scores.append(out[:, 1].cpu().numpy())
                adp_labels.append(y.numpy())
        adp_scores = np.concatenate(adp_scores)
        adp_labels = np.concatenate(adp_labels)
        adp_preds  = (adp_scores > 0.5).astype(int)
        adp_met    = full_metrics(adp_labels, adp_preds, adp_scores)
        logger.info(f'  Adapted model DFE-test: AUC={adp_met["auc"]:.4f}  '
                    f'F1={adp_met["f1"]:.4f}  bal={adp_met["bal_acc"]:.4f}')
        best_val_met['adapter_dfe_test_auc'] = adp_met['auc']
        best_val_met['adapter_dfe_test_f1']  = adp_met['f1']
        del adapted, base_model
        torch.cuda.empty_cache()

    # E16 temperature calibration
    if cfg.get('temp_calibration', False):
        logger.info(f'  Running temperature calibration (E16)...')
        calib_model = RawNet(d_args, DEVICE).to(DEVICE)
        calib_wpath = os.path.join(str(CKPT_DIR), f'{eid}_best_weights.pth')
        calib_model.load_state_dict(torch.load(calib_wpath, map_location=DEVICE, weights_only=True))
        calib_results = calibrate_and_evaluate(
            calib_model, dfe_val_loader, dfe_test_loader, DEVICE, eid, str(CKPT_DIR)
        )
        best_val_met['temp_before_ece'] = calib_results['before']['ece']
        best_val_met['temp_after_ece']  = calib_results['after']['ece']
        best_val_met['temp_after_auc']  = calib_results['after']['auc']
        best_val_met['temperature_T']   = calib_results['after']['temp']
        del calib_model
        torch.cuda.empty_cache()

    elapsed = time.time() - exp_start
    logger.info(f'[{cfg["name"]}] Finished in {elapsed/60:.1f} min')

    save_exp_results(str(CKPT_DIR), eid, best_val_met, history, best_genome)
    completed_results[eid] = load_exp_results(str(CKPT_DIR), eid)
    logger.info(f'Results saved → checkpoints/{eid}_results.json')

logger.info('')
logger.info(f'All experiments: {len(completed_results)}/{len(EXPERIMENTS)} completed.')

In [ ]:
## Evaluate Best Models on Test Set (Deepfake-Eval-2024)
##                      on Test Set (LA)
##                      on Test Set (In the Wild)
# ── Evaluate Best Models on All Test Sets ─────────────────────────────────────
# For each experiment: loads best_weights.pth and evaluates on:
#   1. dfe_full  — 100% Deepfake-Evals-2024 (primary benchmark)
#      EXCEPTION: E13 uses dfe_test (85%) as its primary benchmark to avoid
#                 fitness-set overlap.
#   2. la        — LA test set
#   3. itw       — In-the-Wild test set

TEST_LOADERS = {
    'dfe_full': dfe_full_loader,
    'la':       la_loader,
    'itw':      itw_loader,
}

all_test_results = {}   # exp_id → {dataset_key → metrics}

for cfg in EXPERIMENTS:
    eid     = cfg['id']
    wpath   = os.path.join(str(CKPT_DIR), f'{eid}_best_weights.pth')
    all_test_results[eid] = {}

    if not os.path.exists(wpath):
        logger.warning(f'[{cfg["name"]}] No weights file — skipping test eval.')
        continue

    # ── For E13: primary test is dfe_test (85%), not dfe_full ─────────────
    if cfg.get('dfe_val_fitness', False):
        exp_test_loaders = {
            'dfe_test': dfe_test_loader,   # 85% held-out, no overlap with fitness set
            'la':       la_loader,
            'itw':      itw_loader,
        }
        logger.info(f'[{cfg["name"]}] Using dfe_test (85%) as primary benchmark.')
    else:
        exp_test_loaders = TEST_LOADERS

    # Load model once, evaluate on all test sets
    model = RawNet(d_args, DEVICE).to(DEVICE)
    model.load_state_dict(torch.load(wpath, map_location=DEVICE, weights_only=True))
    model.eval()

    for ds_key, loader in exp_test_loaders.items():
        existing = load_test_results(str(CKPT_DIR), eid, ds_key)
        if existing:
            all_test_results[eid][ds_key] = existing
            logger.info(f'[{cfg["name"]}] [{ds_key}] Already evaluated — '
                        f'AUC={existing["auc"]:.4f}')
            continue

        logger.info(f'[{cfg["name"]}] Evaluating on {ds_key}...')
        all_scores, all_labels = [], []
        with torch.no_grad():
            for x, y in tqdm(loader, desc=f'  {eid} / {ds_key}', leave=False):
                out = model(x.to(DEVICE), is_test=True)
                all_scores.append(out[:, 1].cpu().numpy())
                all_labels.append(y.numpy())

        scores = np.concatenate(all_scores)
        labels = np.concatenate(all_labels)
        preds  = (scores > 0.5).astype(int)

        tm = full_metrics(labels, preds, scores)
        tm['scores'] = scores.tolist()
        tm['labels'] = labels.tolist()

        all_test_results[eid][ds_key] = tm
        save_test_results(str(CKPT_DIR), eid, ds_key, tm)
        logger.info(f'  [{ds_key}] AUC={tm["auc"]:.4f}  bal={tm["bal_acc"]:.4f}  '
                    f'F1={tm["f1"]:.4f}  acc={tm["accuracy"]:.4f}')

    del model
    if str(DEVICE) == 'cuda': torch.cuda.empty_cache()

# ── Summary table ─────────────────────────────────────────────────────────────
logger.info('')
logger.info(f'{"Experiment":<26} {"Pri AUC":>8} {"Pri F1":>7} {"LA AUC":>7} {"LA F1":>6} {"ITW AUC":>8} {"ITW F1":>7}')
logger.info('─' * 85)
for cfg in EXPERIMENTS:
    eid = cfg['id']
    r   = all_test_results.get(eid, {})
    # Primary benchmark key differs for E13
    pri = 'dfe_test' if cfg.get('dfe_val_fitness', False) else 'dfe_full'
    def _g(ds, k): return f'{r[ds][k]:.4f}' if ds in r and k in r[ds] else '  —   '
    logger.info(f'{eid:<26} {_g(pri,"auc"):>8} {_g(pri,"f1"):>7} '
                f'{_g("la","auc"):>7} {_g("la","f1"):>6} '
                f'{_g("itw","auc"):>8} {_g("itw","f1"):>7}')

logger.info(f'Test evaluation complete: {sum(bool(v) for v in all_test_results.values())}/{len(EXPERIMENTS)} models.')

In [ ]:
## Visualization
# ── Plot style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.color': '#e0e0e0', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 8, 'legend.framealpha': 0.9,
})

# Assign each experiment a distinct color
PALETTE = plt.cm.tab10.colors
EXP_COLOR = {cfg['id']: PALETTE[i % 10] for i, cfg in enumerate(EXPERIMENTS)}
GROUP_STYLE = {
    'v2 — Integrated':    {'ls': '-',  'marker': 'o'},
    'v3 — Genome':        {'ls': '--', 'marker': 's'},
    'v3 — Fitness':       {'ls': ':',  'marker': '^'},
    'v3 — Val split':     {'ls': '-.',  'marker': 'D'},
    'v3 — Combined':      {'ls': '--', 'marker': 'P'},
    'v3 — Architecture':  {'ls': ':',  'marker': 'X'},
    'v3 — Calibration':   {'ls': '-',  'marker': 'v'},
    'v3 — Diversity':     {'ls': '--', 'marker': '*'},
}

def group_style(grp):
    return GROUP_STYLE.get(grp, {'ls': '-', 'marker': 'o'})


def fig_fitness_curves():
    """Plot best + mean fitness and best AUC per generation for every experiment."""
    n = len(completed_results)
    if n == 0: return
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
    axes = np.array(axes).flatten()

    for ax_i, cfg in enumerate(EXPERIMENTS):
        eid = cfg['id']
        if eid not in completed_results: continue
        h   = completed_results[eid]['history']
        ax  = axes[ax_i]
        gens = list(range(1, len(h['best_fit']) + 1))
        clr = EXP_COLOR[eid]

        ax.plot(gens, h['best_fit'],  color=clr, lw=2,   label='Best fitness')
        ax.plot(gens, h['mean_fit'],  color=clr, lw=1.2, ls='--', alpha=0.7, label='Mean fitness')
        ax.plot(gens, h['best_auc'],  color='gray', lw=1.5, ls=':',  label='Best AUC (val)')
        ax.set_title(f'{cfg["name"]}', fontweight='bold')
        ax.set_xlabel('Generation')
        ax.set_ylabel('Score')
        ax.legend(loc='lower right')

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Fitness Evolution per Experiment', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig1_fitness_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 1 saved: fitness curves.')


def fig_val_metrics_evolution():
    """Val balanced-acc, F1, recall per generation for every experiment."""
    n = len(completed_results)
    if n == 0: return
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.5))
    axes = np.array(axes).flatten()

    for ax_i, cfg in enumerate(EXPERIMENTS):
        eid = cfg['id']
        if eid not in completed_results: continue
        h   = completed_results[eid]['history']
        ax  = axes[ax_i]
        gens = list(range(1, len(h['best_fit']) + 1))
        clr = EXP_COLOR[eid]

        ax.plot(gens, h['best_bal'],  color=clr,      lw=2,   label='Bal. acc')
        ax.plot(gens, h['best_f1'],   color='steelblue', lw=1.5, ls='--', label='F1')
        ax.plot(gens, h['best_rec'],  color='coral',  lw=1.2, ls=':',  label='Recall')
        ax.plot(gens, h['best_prec'], color='seagreen', lw=1.2, ls='-.', label='Precision')
        ax.set_title(cfg['name'], fontweight='bold')
        ax.set_xlabel('Generation')
        ax.set_ylabel('Metric')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='best')

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Validation Metric Evolution per Experiment', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig2_val_metrics.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 2 saved: val metric curves.')


def fig_test_comparison():
    """Grouped bar chart of test metrics for all experiments (primary benchmark)."""
    if not all_test_results: return
    metrics_shown = ['auc', 'bal_acc', 'accuracy', 'f1', 'precision', 'recall']
    metric_labels = ['AUC', 'Bal Acc', 'Accuracy', 'F1', 'Precision', 'Recall']

    def _pri(cfg):
        """Primary benchmark key for this experiment."""
        return 'dfe_test' if cfg.get('dfe_val_fitness', False) else 'dfe_full'

    exp_list = [cfg for cfg in EXPERIMENTS
                if cfg['id'] in all_test_results and _pri(cfg) in all_test_results[cfg['id']]]
    names    = [cfg['name'] for cfg in exp_list]
    n_exp    = len(exp_list)
    n_met    = len(metrics_shown)

    x = np.arange(n_exp)
    w = 0.12
    offsets = np.linspace(-(n_met-1)/2, (n_met-1)/2, n_met) * w

    fig, ax = plt.subplots(figsize=(max(16, n_exp * 1.8), 5))
    bar_colors = plt.cm.Set2.colors

    for mi, (mkey, mlabel) in enumerate(zip(metrics_shown, metric_labels)):
        vals = [all_test_results[cfg['id']][_pri(cfg)][mkey] for cfg in exp_list]
        ax.bar(x + offsets[mi], vals, w * 0.9, label=mlabel,
               color=bar_colors[mi % len(bar_colors)], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=35, ha='right')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='red', ls='--', lw=0.8, alpha=0.5, label='Random baseline')
    ax.legend(loc='upper right', ncol=4)
    ax.set_title('Test Set Metrics — All Experiments (primary benchmark)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig3_test_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 3 saved: test comparison bars.')


def fig_roc_curves():
    """Overlaid ROC curves on primary test benchmark."""
    if not all_test_results: return
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random (AUC=0.50)')

    for cfg in EXPERIMENTS:
        eid = cfg['id']
        pri = 'dfe_test' if cfg.get('dfe_val_fitness', False) else 'dfe_full'
        if eid not in all_test_results or pri not in all_test_results[eid]:
            continue
        tm = all_test_results[eid][pri]
        if 'scores' not in tm or 'labels' not in tm:
            continue
        scores = np.array(tm['scores'])
        labels = np.array(tm['labels'])
        fpr, tpr, _ = roc_curve(labels, scores)
        auc_val = sk_auc(fpr, tpr)
        gs = group_style(cfg['group'])
        ax.plot(fpr, tpr, color=EXP_COLOR[eid],
                lw=1.8, ls=gs['ls'],
                label=f'{cfg["name"]} (AUC={auc_val:.4f})')

    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves — Primary Test Benchmark', fontweight='bold')
    ax.legend(loc='lower right', fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig4_roc_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 4 saved: ROC curves.')


def fig_confusion_matrices():
    """Grid of confusion matrices for all experiments."""
    if not all_test_results: return

    def _pri(cfg):
        return 'dfe_test' if cfg.get('dfe_val_fitness', False) else 'dfe_full'

    exp_list = [cfg for cfg in EXPERIMENTS
                if cfg['id'] in all_test_results and _pri(cfg) in all_test_results[cfg['id']]]
    n = len(exp_list)
    ncols = min(4, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.2))
    axes = np.array(axes).flatten()

    for i, cfg in enumerate(exp_list):
        ax  = axes[i]
        tm  = all_test_results[cfg['id']][_pri(cfg)]
        cm  = np.array(tm['confusion_matrix'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Pred Real', 'Pred Fake'],
                    yticklabels=['Act Real',  'Act Fake'],
                    cbar=False, linewidths=0.5)
        ax.set_title(cfg['name'], fontsize=8, fontweight='bold')
        ax.set_xlabel(f'AUC={tm["auc"]:.3f}  bal={tm["bal_acc"]:.3f}', fontsize=7.5)

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Confusion Matrices — Primary Test Benchmark',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig5_confusion_matrices.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 5 saved: confusion matrices.')


def fig_diversity_and_cka():
    """Population diversity + mean AUC across generations."""
    n = len(completed_results)
    if n == 0: return
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.2))
    axes = np.array(axes).flatten()

    for ax_i, cfg in enumerate(EXPERIMENTS):
        eid = cfg['id']
        if eid not in completed_results: continue
        h   = completed_results[eid]['history']
        ax  = axes[ax_i]
        gens = list(range(1, len(h['best_fit']) + 1))
        clr = EXP_COLOR[eid]

        ax2 = ax.twinx()
        ax.plot(gens, h['diversity'], color=clr, lw=2, label='Population diversity')
        if 'mean_auc' in h:
            ax2.plot(gens, h['mean_auc'], color='dimgray', lw=1.2, ls='--',
                     label='Mean pop AUC')
        ax.set_title(cfg['name'], fontweight='bold')
        ax.set_xlabel('Generation')
        ax.set_ylabel('Diversity (genome std)', color=clr)
        ax2.set_ylabel('Mean pop AUC', color='dimgray')
        lines1, labs1 = ax.get_legend_handles_labels()
        lines2, labs2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labs1 + labs2, loc='best', fontsize=7.5)

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Population Diversity & Mean AUC per Generation',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig6_diversity.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 6 saved: diversity curves.')


logger.info('All plotting functions defined.')

In [ ]:
## Generate All Figures
logger.info('Generating Figure 1: Fitness evolution curves...')
fig_fitness_curves()

logger.info('Generating Figure 2: Validation metric evolution...')
fig_val_metrics_evolution()

logger.info('Generating Figure 3: Test set comparison bars...')
fig_test_comparison()

logger.info('Generating Figure 4: ROC curves...')
fig_roc_curves()

logger.info('Generating Figure 5: Confusion matrices...')
fig_confusion_matrices()

logger.info('Generating Figure 6: Diversity + mean AUC...')
fig_diversity_and_cka()

In [ ]:
## Final Summary
print()
print('=' * 120)
print(f'  FINAL RESULTS SUMMARY — MeGA-IA Experiments vs Primary Test Benchmarks')
print('=' * 120)
print(f'  {"Experiment":<30} {"Group":<22} {"Benchmark":<11} {"AUC":>6} {"BalAcc":>7} {"Acc":>6} '
      f'{"F1":>6} {"Prec":>6} {"Rec":>6}  {"Val AUC":>8}')
print('─' * 120)

def _pri_key(cfg):
    return 'dfe_test' if cfg.get('dfe_val_fitness', False) else 'dfe_full'

ranked = sorted(
    [cfg for cfg in EXPERIMENTS
     if cfg['id'] in all_test_results and _pri_key(cfg) in all_test_results[cfg['id']]],
    key=lambda c: all_test_results[c['id']][_pri_key(c)]['auc'], reverse=True
)

RANK_ICONS = ['🥇', '🥈', '🥉']
for rank, cfg in enumerate(ranked):
    eid  = cfg['id']
    pkey = _pri_key(cfg)
    tm   = all_test_results[eid][pkey]
    vm   = completed_results.get(eid, {}).get('best_val_metrics', {})
    icon = RANK_ICONS[rank] if rank < 3 else f'  {rank+1}.'
    print(f'{icon} {cfg["name"]:<28} {cfg["group"]:<22} {pkey:<11} '
          f'{tm["auc"]:>6.4f} {tm["bal_acc"]:>7.4f} {tm["accuracy"]:>6.4f} '
          f'{tm["f1"]:>6.4f} {tm["precision"]:>6.4f} {tm["recall"]:>6.4f}  '
          f'{vm.get("auc", float("nan")):>8.4f}')

print('─' * 120)

if ranked:
    best = ranked[0]
    bm   = all_test_results[best['id']][_pri_key(best)]
    print()
    print('🏆 BEST EXPERIMENT:', best['name'])
    print(f'   Config : {best["description"]}')
    print(f'   Test   : AUC={bm["auc"]:.4f}  BalAcc={bm["bal_acc"]:.4f}  '
          f'Acc={bm["accuracy"]:.4f}  F1={bm["f1"]:.4f}')
    print(f'   TP={bm["tp"]}  TN={bm["tn"]}  FP={bm["fp"]}  FN={bm["fn"]}')

# ── Per-group best ────────────────────────────────────────────────────────────
print()
print('── PER-GROUP BEST PERFORMANCE ───────────────────────────────────────────')
groups = {}
for cfg in EXPERIMENTS:
    eid  = cfg['id']
    pkey = _pri_key(cfg)
    if eid not in all_test_results or pkey not in all_test_results[eid]:
        continue
    grp     = cfg['group']
    auc_val = all_test_results[eid][pkey]['auc']
    if grp not in groups or auc_val > groups[grp]['auc']:
        groups[grp] = {'name': cfg['name'], 'auc': auc_val,
                       'bal': all_test_results[eid][pkey]['bal_acc'],
                       'f1':  all_test_results[eid][pkey]['f1']}
for grp, info in groups.items():
    print(f'  {grp:<28}  best={info["name"]:<30}  '
          f'AUC={info["auc"]:.4f}  bal={info["bal"]:.4f}  F1={info["f1"]:.4f}')

# ── E15/E16 special results ───────────────────────────────────────────────────
calib_path = os.path.join(str(CKPT_DIR), 'E16_temp_calib_calibration.json')
if os.path.exists(calib_path):
    with open(calib_path) as f:
        cr = json.load(f)
    print()
    print('── E16 TEMPERATURE CALIBRATION RESULTS ─────────────────────────────────')
    print(f'  Before: AUC={cr["before"]["auc"]:.4f}  ECE={cr["before"]["ece"]:.4f}  T=1.000')
    print(f'  After : AUC={cr["after"]["auc"]:.4f}  ECE={cr["after"]["ece"]:.4f}  T={cr["after"]["temp"]:.3f}')

# ── Checkpoint files ──────────────────────────────────────────────────────────
print()
print('── CHECKPOINTED FILES ────────────────────────────────────────────────────')
for f in sorted(os.listdir(str(CKPT_DIR))):
    fpath = os.path.join(str(CKPT_DIR), f)
    print(f'  {f:<60}  {os.path.getsize(fpath)/1e3:>8.1f} KB')

print()
print('=' * 120)
print('  All figures saved to:', str(CKPT_DIR))
print('=' * 120)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2: Parent baselines + top-experiment re-runs on DFE-val fitness
# ══════════════════════════════════════════════════════════════════════════════
import json, os, time
import numpy as np
import torch
from tqdm import tqdm

P2_DIR = os.path.join(str(CKPT_DIR), "phase2")
os.makedirs(P2_DIR, exist_ok=True)

# ── Helper: evaluate any model on any loader ──────────────────────────────────
def evaluate_on_loader(model, loader, device, label):
    model.eval()
    all_scores, all_labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=f'  {label}', leave=False):
            out = model(x.to(device), is_test=True)
            all_scores.append(out[:, 1].cpu().numpy())
            all_labels.append(y.numpy())
    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels)
    preds  = (scores > 0.5).astype(int)
    met    = full_metrics(labels, preds, scores)
    met['scores'] = scores.tolist()
    met['labels'] = labels.tolist()
    logger.info(f'  [{label}] AUC={met["auc"]:.4f}  bal={met["bal_acc"]:.4f}  '
                f'F1={met["f1"]:.4f}  acc={met["accuracy"]:.4f}')
    return met

phase2_results = {}   # key → {dataset → metrics}

# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — Parent model baselines on LA + ITW
# ══════════════════════════════════════════════════════════════════════════════
logger.info('')
logger.info('═' * 70)
logger.info('PART 1 — Parent model baselines (LA + ITW)')
logger.info('═' * 70)

for parent_label, parent_state in [('parent_LA', p1_state), ('parent_ITW', p2_state)]:
    cache_path = os.path.join(P2_DIR, f'{parent_label}_results.json')
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            phase2_results[parent_label] = json.load(f)
        logger.info(f'[{parent_label}] Loaded from cache.')
        for ds, m in phase2_results[parent_label].items():
            logger.info(f'  [{ds}] AUC={m["auc"]:.4f}  bal={m["bal_acc"]:.4f}  F1={m["f1"]:.4f}')
        continue

    logger.info(f'Evaluating {parent_label}...')
    model = RawNet(d_args, DEVICE).to(DEVICE)
    model.load_state_dict(parent_state)

    res = {}
    for ds_label, loader in [('la_test', la_loader), ('itw_test', itw_loader)]:
        res[ds_label] = {k: v for k, v in evaluate_on_loader(model, loader, DEVICE, ds_label).items()
                         if k not in ('scores', 'labels')}

    del model
    torch.cuda.empty_cache()

    phase2_results[parent_label] = res
    with open(cache_path, 'w') as f:
        json.dump(res, f, indent=2)
    logger.info(f'[{parent_label}] saved → {cache_path}')

# ══════════════════════════════════════════════════════════════════════════════
# PART 2 — Select experiments to re-run: E01 + top 3 others by primary AUC
# ══════════════════════════════════════════════════════════════════════════════
logger.info('')
logger.info('═' * 70)
logger.info('PART 2 — Selecting top experiments for Phase 2 re-run')
logger.info('═' * 70)

def _pri_key(cfg):
    return 'dfe_test' if cfg.get('dfe_val_fitness', False) else 'dfe_full'

# Score every completed experiment by its primary-benchmark AUC
scored = []
for cfg in EXPERIMENTS:
    eid  = cfg['id']
    pkey = _pri_key(cfg)
    if eid in all_test_results and pkey in all_test_results[eid]:
        scored.append((all_test_results[eid][pkey]['auc'], cfg))
scored.sort(key=lambda x: x[0], reverse=True)

# Always include E00 (doc config) and E01; add top-2 others by AUC
e00_cfg = next(c for c in EXPERIMENTS if c['id'] == 'E00_mega_ia_doc')
e01_cfg = next(c for c in EXPERIMENTS if c['id'] == 'E01_mega_ia_v2')
always_ids  = {'E00_mega_ia_doc', 'E01_mega_ia_v2'}
others_top2 = [cfg for _, cfg in scored if cfg['id'] not in always_ids][:2]
phase2_exps = [e00_cfg, e01_cfg] + others_top2

logger.info('Phase 2 experiments selected:')
for cfg in phase2_exps:
    pkey = _pri_key(cfg)
    auc  = all_test_results.get(cfg['id'], {}).get(pkey, {}).get('auc', float('nan'))
    logger.info(f'  {cfg["id"]:<26}  phase1_AUC={auc:.4f}')

# ══════════════════════════════════════════════════════════════════════════════
# PART 3 — Re-run selected experiments with DFE-val fitness
# ══════════════════════════════════════════════════════════════════════════════
logger.info('')
logger.info('═' * 70)
logger.info('PART 3 — Phase 2 evolution (fitness=DFE-val 15%)')
logger.info('═' * 70)

phase2_evo_results = {}   # exp_id → load_exp_results dict

for cfg in phase2_exps:
    orig_id   = cfg['id']
    p2_id     = f'P2_{orig_id}'
    p2_wpath  = os.path.join(P2_DIR, f'{p2_id}_best_weights.pth')

    # Build a Phase-2 config: same hyperparams, DFE-val fitness, new id
    p2_cfg = dict(cfg)
    p2_cfg['id']              = p2_id
    p2_cfg['name']            = f'P2 {cfg["name"]}'
    p2_cfg['dfe_val_fitness'] = True   # always use DFE-val for fitness

    # Check for existing result
    res_path = os.path.join(P2_DIR, f'{p2_id}_results.json')
    if os.path.exists(res_path):
        with open(res_path) as f:
            phase2_evo_results[p2_id] = json.load(f)
        logger.info(f'[{p2_id}] Already done — loaded from cache.')
        bvm = phase2_evo_results[p2_id]['best_val_metrics']
        logger.info(f'  val AUC={bvm["auc"]:.4f}  bal={bvm["bal_acc"]:.4f}  F1={bvm["f1"]:.4f}')
        continue

    logger.info(f'')
    logger.info(f'Starting [{p2_cfg["name"]}]  (fitness on DFE-val 15%)')
    t0 = time.time()

    best_val_met, history, best_genome = run_mega_ia(
        p2_cfg, p1_state, p2_state, d_args, DEVICE,
        dfe_val_loader,   # fitness loader = DFE val (15%)
        p1_feats_dfe,     # p1 features extracted on DFE-val
        P2_DIR,
        device2=DEVICE2
    )

    logger.info(f'[{p2_cfg["name"]}] Finished in {(time.time()-t0)/60:.1f} min')

    # Save evolution results
    out = {
        'exp_id': p2_id,
        'best_val_metrics': {k: v for k, v in best_val_met.items() if k not in ('scores','labels')},
        'history':     history,
        'best_genome': best_genome.tolist() if hasattr(best_genome, 'tolist') else best_genome,
    }
    with open(res_path, 'w') as f:
        json.dump(out, f, indent=2, default=str)
    phase2_evo_results[p2_id] = out
    logger.info(f'Results saved → {res_path}')

# ══════════════════════════════════════════════════════════════════════════════
# PART 4 — Test Phase 2 models on DFE-test (85%), LA, ITW
# ══════════════════════════════════════════════════════════════════════════════
logger.info('')
logger.info('═' * 70)
logger.info('PART 4 — Phase 2 test evaluation (DFE-test 85% + LA + ITW)')
logger.info('═' * 70)

phase2_test_results = {}   # p2_id → {ds → metrics}

for cfg in phase2_exps:
    orig_id  = cfg['id']
    p2_id    = f'P2_{orig_id}'
    p2_wpath = os.path.join(P2_DIR, f'{p2_id}_best_weights.pth')

    test_cache = os.path.join(P2_DIR, f'{p2_id}_test_results.json')
    if os.path.exists(test_cache):
        with open(test_cache) as f:
            phase2_test_results[p2_id] = json.load(f)
        logger.info(f'[{p2_id}] Test results loaded from cache.')
        for ds, m in phase2_test_results[p2_id].items():
            logger.info(f'  [{ds}] AUC={m["auc"]:.4f}  bal={m["bal_acc"]:.4f}  F1={m["f1"]:.4f}')
        continue

    if not os.path.exists(p2_wpath):
        logger.warning(f'[{p2_id}] No weights file — skipping test eval.')
        continue

    logger.info(f'Testing [{p2_id}]...')
    model = RawNet(d_args, DEVICE).to(DEVICE)
    model.load_state_dict(torch.load(p2_wpath, map_location=DEVICE, weights_only=True))

    res = {}
    for ds_label, loader in [
        ('dfe_test_85pct', dfe_test_loader),
        ('la_test',        la_loader),
        ('itw_test',       itw_loader),
    ]:
        m = evaluate_on_loader(model, loader, DEVICE, ds_label)
        res[ds_label] = {k: v for k, v in m.items() if k not in ('scores', 'labels')}

    del model
    torch.cuda.empty_cache()

    phase2_test_results[p2_id] = res
    with open(test_cache, 'w') as f:
        json.dump(res, f, indent=2)
    logger.info(f'[{p2_id}] Test results saved → {test_cache}')

# ══════════════════════════════════════════════════════════════════════════════
# PART 5 — Summary
# ══════════════════════════════════════════════════════════════════════════════
logger.info('')
print()
print('=' * 130)
print('  PHASE 2 SUMMARY')
print('=' * 130)

# ── Parent baselines ──────────────────────────────────────────────────────────
print(f'\n{"── PARENT BASELINES":-<130}')
print(f'  {"Model":<20} {"LA AUC":>8} {"LA F1":>7} {"LA bal":>7} {"ITW AUC":>9} {"ITW F1":>8} {"ITW bal":>8}')
print('  ' + '─' * 70)
for plabel in ['parent_LA', 'parent_ITW']:
    r = phase2_results.get(plabel, {})
    def _g(ds, k): return f'{r[ds][k]:.4f}' if ds in r else '   —  '
    print(f'  {plabel:<20} '
          f'{_g("la_test","auc"):>8} {_g("la_test","f1"):>7} {_g("la_test","bal_acc"):>7} '
          f'{_g("itw_test","auc"):>9} {_g("itw_test","f1"):>8} {_g("itw_test","bal_acc"):>8}')

# ── Phase 2 experiment results ────────────────────────────────────────────────
print(f'\n{"── PHASE 2 EXPERIMENTS (fitness=DFE-val 15%)":-<130}')
print(f'  {"Experiment":<28} {"Val AUC":>8} '
      f'{"DFEtest AUC":>12} {"DFEtest F1":>11} '
      f'{"LA AUC":>8} {"LA F1":>7} '
      f'{"ITW AUC":>9} {"ITW F1":>8}')
print('  ' + '─' * 100)

ranked_p2 = sorted(
    [(cfg, f'P2_{cfg["id"]}') for cfg in phase2_exps],
    key=lambda x: phase2_test_results.get(x[1], {}).get('dfe_test_85pct', {}).get('auc', -1),
    reverse=True
)

RANK_ICONS = ['🥇', '🥈', '🥉', '  4.']
for rank, (cfg, p2_id) in enumerate(ranked_p2):
    bvm = phase2_evo_results.get(p2_id, {}).get('best_val_metrics', {})
    tr  = phase2_test_results.get(p2_id, {})
    def _g(ds, k): return f'{tr[ds][k]:.4f}' if ds in tr and k in tr[ds] else '    —  '
    icon = RANK_ICONS[rank] if rank < len(RANK_ICONS) else f'  {rank+1}.'
    print(f'{icon} {cfg["id"]:<26} '
          f'{bvm.get("auc", float("nan")):>8.4f} '
          f'{_g("dfe_test_85pct","auc"):>12} {_g("dfe_test_85pct","f1"):>11} '
          f'{_g("la_test","auc"):>8} {_g("la_test","f1"):>7} '
          f'{_g("itw_test","auc"):>9} {_g("itw_test","f1"):>8}')

# ── Phase 1 vs Phase 2 delta for E01 ─────────────────────────────────────────
print(f'\n{"── PHASE 1 → PHASE 2 DELTA (E01)":-<130}')
p1_e01 = all_test_results.get('E01_mega_ia_v2', {}).get('dfe_full', {})
p2_e01 = phase2_test_results.get('P2_E01_mega_ia_v2', {}).get('dfe_test_85pct', {})
if p1_e01 and p2_e01:
    for metric in ['auc', 'bal_acc', 'f1', 'accuracy']:
        delta = p2_e01.get(metric, 0) - p1_e01.get(metric, 0)
        arrow = '▲' if delta > 0 else '▼'
        print(f'  {metric:<12}  Phase1={p1_e01[metric]:.4f}  Phase2={p2_e01[metric]:.4f}  '
              f'{arrow} {abs(delta):.4f}')

# ── Checkpoint files ──────────────────────────────────────────────────────────
print(f'\n{"── PHASE 2 CHECKPOINT FILES":-<130}')
for fname in sorted(os.listdir(P2_DIR)):
    fpath = os.path.join(P2_DIR, fname)
    print(f'  {fname:<65}  {os.path.getsize(fpath)/1e3:>8.1f} KB')

print()
print('=' * 130)
logger.info('Phase 2 complete.')